# Abertura P × Q — Enel Distribuição São Paulo

Este notebook abre a receita de fornecimento de energia por item tarifário e por família, a partir
da tarifa homologada pela ANEEL. O manual gerencial de cada passo, com o porquê de cada
transformação, está em [`procedimentos/PROCEDIMENTO_PXQ.md`](procedimentos/PROCEDIMENTO_PXQ.md);
as normas que fundamentam o cálculo estão na seção 3 do [README](../README.md).

A saída traz dois blocos de valores lado a lado:

| Bloco | O que é | Para que serve |
|---|---|---|
| **1 — Calculado** | quantidade × tarifa homologada cheia, nos 37 itens e nas 4 camadas | É a régua: mostra o que a tarifa explica sozinha |
| **2 — Ajustado** | o bloco 1 corrigido até fechar na receita faturada | É o produto final: a receita aberta por item |

O bloco 1 não fecha com a receita porque a tabela de tarifas traz a tarifa cheia e muitos clientes
pagam com desconto. O bloco 2 devolve cada desconto aos itens sobre os quais a regulação o faz
incidir — encargos do APE, Fio A e Fio B na fonte incentivada — e só o que sobra é distribuído por
igual.

**O caminho:**

- **Parte I — Fornecimento (BW).** Ler a extração → separar o que é tarifa → montar a chave →
  buscar a tarifa → definir a quantidade → decidir quais parcelas incidem → multiplicar (bloco 1) →
  tarifa proporcional → agrupar e estimar as faixas dos benefícios sociais → ajustar (bloco 2) →
  conferir.
- **Parte II — Geradoras, suprimentos e uso da distribuição (SD).** Classificar os clientes → ler a
  planilha → abrir a receita → juntar com o BW e conferir → gravar a planilha.

---
## Passo 0 — Parâmetros

Tudo o que muda de um mês para outro está nesta célula; o restante do notebook não é editado. Todo
mês muda só `REFERENCIA`. Os demais parâmetros (arquivos de origem, 2ª passada, regime
APE, tarifa proporcional, peso manual por subgrupo) estão descritos no Passo 0 do procedimento.

In [ ]:
import os
import re
import unicodedata
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl

# ---------------------------------------------------------------- pastas
BASE = Path(os.environ.get("PXQ_BASE", r"C:\python_projects\claude\p_x_q\SP\BW"))

# Tabela central de tarifas homologadas (TA, BE, CVA, FIN por vigencia), gerada por
# 3-Py/1-Entrega/extrai_pcat.py a partir dos PCATs. Adicionar uma vigencia nova e so
# acrescentar uma linha nesse script e rodar de novo - nada aqui precisa mudar.
ARQ_TARIFA_LONG = BASE / "tbl" / "tarifas" / "tarifa_long.csv"
ARQ_SD              = BASE / "src" / "SD_ENEL SP_completo.xlsx"

# ---------------------------------------------------------------- periodo
REFERENCIA = "2026-06"

# BW revenue base of the month: parquet prepared from the original csv by
# 3-Py/1-Entrega/prepara_base_bw.py (src/csv_original -> src/parquet)
ARQ_BW = BASE / "src" / "parquet" / f"{REFERENCIA[5:7]}-{REFERENCIA[:4]}_ENEL SP_RECEITA.parquet"

# Excecao por subgrupo (opcional): pares (dias na vigencia mais antiga, dias na mais
# recente), para quando um subgrupo fatura em ciclo diferente do resto do mes. So tem
# efeito quando o mes toca exatamente duas vigencias (caso normal de reajuste). No caso
# comum, o peso de cada vigencia e calculado sozinho pela sobreposicao de dias entre
# VIGENCIA_INICIO/VIGENCIA_FIM (em tbl/tarifas/tarifa_long.csv) e o mes de REFERENCIA -
# nao ha mais variavel de dias para setar a mao todo mes.
# Ex.: {"B1": (19, 12), "B3": (20, 11), "A4": (28, 3)}
DIAS_POR_SUBGRUPO = {}

# 2a passada (fechamento de conta, Passo 9). True = o bloco 2 fecha exatamente na receita do BW
# (entrega padrao). False = o bloco 2 para na 1a passada; a diferenca que sobra NAO e rateada e
# vai para as abas "Diferencas_linha" (linha a linha do BW) e "Diferencas_grupo" da planilha,
# gravada com o sufixo _sem_2a_passada para nao sobrescrever a entrega padrao.
APLICAR_2A_PASSADA = True

# Regime APE (Passo 9). True = as linhas de isencao de encargos do autoprodutor por equiparacao
# (TUSD - Desconto Consumo ZQC036/181/182 e COMPLEMENTO RES.ANEEL 166/05 ZQC059/183/184) entram na
# base dos clientes livres do grupo A e abatem so os itens da TUSD de energia em que a tarifa APE
# difere da tarifa do livre (encargos). False = essas linhas ficam fora da conta.
REGIME_APE = True

# Tarifa proporcional (Passo 7b). No mes do reajuste e no mes seguinte, o peso de cada vigencia
# por subgrupo e estimado pela receita, calibrado pela relacao receita/calculado dos ultimos
# N_MESES_CALIBRACAO meses validos (1 = so o ultimo mes valido; ate 12). Todo mes roda grava
# tbl/calibracao/calibracao_<REFERENCIA>.csv, que forma esse historico. Sem historico, reajuste
# pequeno ou receita pequena, o subgrupo fica com o peso do calendario. DIAS_POR_SUBGRUPO, se
# preenchido, tem precedencia sobre a estimativa.
CALIBRAR_TARIFA_PROPORCIONAL = True
N_MESES_CALIBRACAO = 3
DIR_CALIBRACAO = BASE / "tbl" / "calibracao"
LIMIAR_DIF_TARIFA = 0.02          # reajuste do subgrupo abaixo disso: estimativa instavel, usa calendario
MIN_RECEITA_CALIBRACAO = 1_000_000  # receita cativa do subgrupo no mes abaixo disso: usa calendario

SAIDA = BASE / "1-Entrega" / "pxq" / (f"pxq_{REFERENCIA}.xlsx" if APLICAR_2A_PASSADA
                                      else f"pxq_{REFERENCIA}_sem_2a_passada.xlsx")

print("extracao :", ARQ_BW.name)
print("saida    :", SAIDA.name)
print("2a passada:", "aplicada" if APLICAR_2A_PASSADA else "DESLIGADA - diferencas em abas proprias")
print("regime APE:", "ligado" if REGIME_APE else "desligado")
print("tarifa proporcional:", f"calibrada pela receita (media de {N_MESES_CALIBRACAO} mes(es))"
      if CALIBRAR_TARIFA_PROPORCIONAL else "so calendario")


---
## Passo 1 — Ler a extração do BW

A base do mês é o parquet `src/parquet/MM-AAAA_ENEL SP_RECEITA.parquet`, preparado a partir do csv
original por `3-Py/1-Entrega/prepara_base_bw.py`. Ela vem detalhada por lote, data de faturamento, mês
de referência e flag de refaturamento; aqui é somada nas dimensões de classe, categoria tarifária,
código estatístico, operação, subclasse e mercado, mais o **tipo de faturamento**:

| Tipo | Regra |
|---|---|
| **CNR** | `Operação principal TEXTO` = "Faturamento Manual de Energia" (TOI - Termo de Ocorrência e Inspeção) |
| **REFAT** | `Flag de Refaturamento` = "X" (estorno e nova conta vêm em pares, ambos marcados) |
| **FAT** | os demais lançamentos |

O tipo segue até a saída e preenche a coluna `TipoFaturamento` do layout Receita Faturada.

A coluna **Montante (R$)** é a receita registrada pelo SAP: é ela que o resultado final precisa
reproduzir. Essa base é a mesma do informe de gestão.


In [ ]:
# Clean parquet column -> name used here. The dimensions are the ones of the former Excel extraction;
# the billing type (FAT / REFAT / CNR) is derived from the new fields; lot, original reference and
# billing date are kept for later steps.
COLUNAS_BW = {
    "Classe principal":            "CLASSE_PRINCIPAL",
    "Categoria tarifa":            "CATEGORIA_TARIFA",
    "Grp.estatísts.mont.":         "GRP_MONT",
    "Grp.estatísts.quant.":        "GRP_QUANT",         # the ZQC/ZQD code
    "Grp.estatísts.quant. TEXTO":  "GRP_QUANT_TXT",     # where the tariff period (posto) comes from
    "Operação principal":          "OPERACAO",
    "Operação principal TEXTO":    "OPERACAO_TXT",
    "Suboperação":                 "SUBOPERACAO",
    "Suboperação TEXTO":           "SUBOPERACAO_TXT",
    "Flag Cli.Liv./ Cat.":         "FLAG_MERCADO",      # Cliente Cativo / Livre / Hibrido
    "Classe/Sub-classe":           "CLASSE_SUBCLASSE",
    "TIPO_FATURAMENTO":            "TIPO_FATURAMENTO",  # derived below: FAT / REFAT / CNR
}
MEDIDAS_BW = {
    "TUSD Faturado (kWh)":         "QTD_CONSUMO_TUSD",  # consumption billed on the TUSD side, kWh
    "TUSD Faturado (kW)":          "QTD_DEMANDA_TUSD",  # demand billed on the TUSD side, kW
    "Demanda Faturada (kW)":       "QTD_DEMANDA",       # billed demand, kW
    "Consumo TE Faturado (kWh)":   "QTD_CONSUMO_TE",    # consumption billed on the TE side, kWh
    "Montante (R$)":               "RECEITA_BRL",       # amount R$ <- comparison base
}

base = pd.read_parquet(ARQ_BW)
# billing type of each line (TipoFaturamento of the Receita Faturada layout):
#   CNR   = TOI, booked as manual energy billing (it wins over the rebilling flag)
#   REFAT = rebilling flag "X" (reversal and new bill come in pairs, both flagged)
#   FAT   = everything else
OPERACAO_TOI = "Faturamento Manual de Energia"
base["TIPO_FATURAMENTO"] = np.select(
    [base["Operação principal TEXTO"].eq(OPERACAO_TOI), base["Flag de Refaturamento"].eq("X")],
    ["CNR", "REFAT"], "FAT")
# same grain as the former Excel extraction: the base is detailed by lot, billing date, original
# reference and rebilling flag; here those are summed over, keeping the billing type
bw = (base.groupby(list(COLUNAS_BW), dropna=False, sort=False)[list(MEDIDAS_BW)].sum()
          .reset_index().rename(columns={**COLUNAS_BW, **MEDIDAS_BW}))
bw = bw[bw["CATEGORIA_TARIFA"] != ""].copy()
# control totals of the area (former Excel columns AF and AG): consumption = TUSD + TE kWh,
# demand = TUSD kW + billed kW
bw["TOTAL_CONSUMO_XLS"] = bw["QTD_CONSUMO_TUSD"] + bw["QTD_CONSUMO_TE"]
bw["TOTAL_DEMANDA_XLS"] = bw["QTD_DEMANDA_TUSD"] + bw["QTD_DEMANDA"]
# line of the aggregated base each record comes from - used later to trace back
bw.insert(0, "LINHA_EXCEL", np.arange(1, len(bw) + 1))
for c in list(MEDIDAS_BW.values()) + ["TOTAL_CONSUMO_XLS", "TOTAL_DEMANDA_XLS"]:
    bw[c] = bw[c].astype(float)

print(f"{len(base):,} linhas na base | {len(bw):,} linhas agregadas | receita total R$ {bw['RECEITA_BRL'].sum():,.2f}")
print("por tipo de faturamento: " + " | ".join(
    f"{t} R$ {v:,.2f}" for t, v in bw.groupby("TIPO_FATURAMENTO")["RECEITA_BRL"].sum().items()))
bw.head(3)

---
## Passo 2 — Separar o que é fornecimento

Só entram no cálculo os lançamentos que têm tarifa homologada por item: **30 códigos estatísticos**
de consumo e demanda (16 de consumo do cativo, inclusive a energia compensada do posto
convencional; 2 de consumo do livre; 12 de demanda).

Tributos, bandeiras, reativos, ultrapassagem, substituição tributária, acertos da CCEE e as linhas de
isenção do APE têm receita, mas não têm tarifa por item: ficam marcados como **não tarifáveis**. Eles
não somem — são gravados linha a linha na saída, para que o total continue igual ao do BW e para que
o conversor da Receita Faturada aproveite tributos e bandeiras. As linhas de isenção do APE são
usadas no Passo 9.

In [ ]:
CODIGOS_TARIFAVEIS = {
    # consumo do cliente cativo - a quantidade vem na linha lancada em TE
    "ZQC145", "ZQC146", "ZQC147", "ZQC148", "ZQC153", "ZQC154",
    "ZQC163", "ZQC164", "ZQC165", "ZQC166", "ZQC167", "ZQC168",
    "ZQC218", "ZQC219", "ZQC225", "ZQC226",
    # consumo do cliente livre - so TUSD
    "ZQC042", "ZQC043",
    # demanda
    # (demand overrun ZQD021/022/023 is a penalty, not a tariff item: kept out, like the free market's)
    "ZQD001", "ZQD003", "ZQD004", "ZQD017", "ZQD018",
    "ZQD036", "ZQD058", "ZQD059", "ZQD060", "ZQD061", "ZQD062", "ZQD063",
}

bw["TARIFAVEL"] = bw["GRP_QUANT"].isin(CODIGOS_TARIFAVEIS)

# grandeza: consumo (codigos ZQC) ou demanda (codigos ZQD)
bw["GRANDEZA"] = np.where(bw["GRP_QUANT"].astype(str).str.startswith("ZQD"), "DEMANDA",
                 np.where(bw["GRP_QUANT"].astype(str).str.startswith("ZQC"), "CONSUMO", "NA"))

# em qual parcela o SAP lancou a linha. Demanda e sempre faturada na TUSD.
def parcela_do_lancamento(row):
    texto = str(row["GRP_MONT"])
    if re.search(r"\bTE\b", texto):
        return "TE"
    if "TUSD" in texto:
        return "TUSD"
    return "TUSD" if row["GRANDEZA"] == "DEMANDA" else "NA"

bw["PARCELA_LANCAMENTO"] = bw.apply(parcela_do_lancamento, axis=1)

resumo = bw.groupby("TARIFAVEL")["RECEITA_BRL"].agg(["size", "sum"])
print(resumo.to_string())
print(f"\nparticipacao nao tarifavel: {100*resumo.loc[False,'sum']/resumo['sum'].sum():.1f}%")

---
## Passo 3 — Montar a chave tarifária

A tarifa não é um número por cliente: é um número por combinação de características do
fornecimento. A chave é a concatenação, **sem separador**, de oito atributos:

```
SUBGRUPO + MODALIDADE + CLASSE + SUBCLASSE + DETALHE + UC + POSTO + UNIDADE
```

Onde não há valor, o texto é literalmente `Não se aplica`. Cada atributo é deduzido da categoria
tarifária, da subclasse ou do código estatístico. Os códigos de consumo por posto horário definem o
posto em qualquer grupo; só indicam a modalidade Branca junto com a categoria `_TB`.

Neste passo a subclasse também é classificada quanto aos **benefícios sociais** (Baixa Renda ou
Desconto Social), usados nos Passos 7 e 8.

In [ ]:
NAO_SE_APLICA = "Não se aplica"

# Codigos de consumo lancados por posto horario (ponta, intermediario, fora de ponta).
# NAO sao exclusivos da tarifa branca: o grupo A usa os mesmos codigos nas tarifas
# horarias Azul e Verde. Eles definem o POSTO; so viram Branca junto com categoria "_TB".
CODIGOS_POSTO_HORARIO = {"ZQC145", "ZQC146", "ZQC147", "ZQC148", "ZQC218", "ZQC219"}
CODIGOS_ILUMINACAO    = {"ZQC167", "ZQC168"}   # sempre B4a, mesmo em categoria B4b
CODIGOS_BAIXA_RENDA   = {"ZQC163", "ZQC164"}   # a propria linha de desconto

# Subclasse (coluna bruta "Classe/Sub-classe" do BW) -> beneficio de isencao de
# encargos setoriais (PRORET Submodulo 7.1, paragrafos 18.II/18.V e 23.III/23.V).
# "None" = sem beneficio. Confirmar/ajustar com a area sempre que aparecer uma
# subclasse nova no BW.
SUBCLASSE_BENEFICIO = {
    "Residencial-Plena":                  None,
    "Residencial-Baixa Renda BPC":        "BAIXA_RENDA",
    "Residencial-Baixa Renda":            "BAIXA_RENDA",
    "Residencial-B Optante":              None,
    "Residencial-Baixa Renda Indígena":   "BAIXA_RENDA",
    "Residencial-Baixa Renda Quilombola": "BAIXA_RENDA",
    "Residencial-Desconto Social":        "DESCONTO_SOCIAL",
}

# Cada beneficio isenta um subconjunto dos componentes de Encargos (FAMILIA["ENCARGOS"]).
# Baixa Renda: alineas d/e/f do TUSD (Quota CDE, PROINFA, CDE COVID/Escassez Hidrica)
#              + alinea f do TE (CDE GD) - sem limite de consumo.
# Desconto Social: alineas d/f do TUSD (sem PROINFA) + alinea f do TE. A norma limita o
#              beneficio aos primeiros 120 kWh; o BW nao traz o consumo por UC, entao a
#              participacao de cada faixa e estimada pela receita no Passo 8.
ITENS_ISENTOS_BENEFICIO = {
    "BAIXA_RENDA":     ["TUSD_CDE", "TUSD_PROINFA", "TUSD_CDE_COVID", "TE_CDE_GD"],
    "DESCONTO_SOCIAL": ["TUSD_CDE", "TUSD_CDE_COVID", "TE_CDE_GD"],
}

PADRAO_SUBGRUPO = re.compile(r"^(A2|A3a|A4|AS|B1|B2|B3|B4a|B4b)")

def sem_acento(texto):
    norm = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in norm if not unicodedata.combining(c)).lower()

def deriva_subgrupo(categoria):
    m = PADRAO_SUBGRUPO.match(str(categoria).strip())
    if not m:
        return None
    token = m.group(1)
    return "B4" if token in ("B4a", "B4b") else token

def deriva_modalidade(categoria, codigo):
    cat = str(categoria)
    if "Azul" in cat:
        return "Azul"
    if "Verde" in cat:
        return "Verde"
    if cat.endswith("_TB") and codigo in CODIGOS_POSTO_HORARIO:
        return "Branca"
    return "Convencional"

def deriva_classe(subgrupo):
    return {"B1": "Residencial", "B2": "Rural",
            "B4": "Iluminação pública"}.get(subgrupo, NAO_SE_APLICA)

def deriva_subclasse(subgrupo, categoria, classe_subclasse, codigo):
    cat, sub = str(categoria), str(classe_subclasse)
    if subgrupo == "B4":
        if codigo in CODIGOS_ILUMINACAO or cat.startswith("B4a"):
            return "Iluminação pública – B4a"
        return "Iluminação pública – B4b"
    if subgrupo != "B1":
        return NAO_SE_APLICA
    if "Desconto Social" in sub:
        return "Residencial" if codigo in CODIGOS_POSTO_HORARIO \
               else "Residencial Desconto Social – faixa 02"
    if "Baixa Renda" in sub:
        if cat.endswith("_TB") or codigo in CODIGOS_BAIXA_RENDA:
            return "Residencial"
        return "Baixa Renda"
    return "Residencial"

def deriva_posto(codigo, texto_quant, texto_mont, subclasse):
    """Prefixo E para consumo e D para demanda; sufixo lido do texto do lancamento."""
    prefixo = "D" if str(codigo).startswith("ZQD") else "E"
    texto = sem_acento(texto_quant) + " " + sem_acento(texto_mont)
    if "intermediari" in texto:
        sufixo = "INT"
    elif "fora ponta" in texto or "fora de ponta" in texto:
        sufixo = "FP"
    elif "ponta" in texto:
        sufixo = "P"
    else:
        sufixo = ""
    posto = prefixo + sufixo
    if posto == "E" and subclasse == "Residencial Desconto Social – faixa 02":
        return NAO_SE_APLICA
    return posto

def deriva_unidade(codigo):
    return "kW" if str(codigo).startswith("ZQD") else "MWh"

def monta_chave(row):
    subgrupo = deriva_subgrupo(row["CATEGORIA_TARIFA"])
    if subgrupo is None:
        return pd.Series([None] * 9)
    modalidade = deriva_modalidade(row["CATEGORIA_TARIFA"], row["GRP_QUANT"])
    classe     = deriva_classe(subgrupo)
    subclasse  = deriva_subclasse(subgrupo, row["CATEGORIA_TARIFA"],
                                  row["CLASSE_SUBCLASSE"], row["GRP_QUANT"])
    detalhe    = NAO_SE_APLICA     # APE, SCEE e tipos so existem em geradoras
    uc         = NAO_SE_APLICA     # unidades nomeadas so existem em geradoras
    posto      = deriva_posto(row["GRP_QUANT"], row["GRP_QUANT_TXT"],
                              row["GRP_MONT"], subclasse)
    unidade    = deriva_unidade(row["GRP_QUANT"])
    atributos  = [subgrupo, modalidade, classe, subclasse, detalhe, uc, posto, unidade]
    return pd.Series(atributos + ["".join(atributos)])

bw[["SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE",
    "UC", "POSTO", "UNIDADE", "CHAVE"]] = bw.apply(monta_chave, axis=1)

print("chaves distintas nas linhas tarifaveis:",
      bw.loc[bw['TARIFAVEL'], 'CHAVE'].nunique())
bw.loc[bw["TARIFAVEL"], ["LINHA_EXCEL", "GRP_QUANT", "CHAVE"]].head(3)

---
## Passo 4 — As tarifas homologadas

As tarifas vêm de uma tabela única, `tbl/tarifas/tarifa_long.csv`: uma linha por vigência × camada ×
chave, com o início e o fim de cada vigência. A tabela é gerada por `3-Py/1-Entrega/extrai_pcat.py` a partir
dos PCATs; para incluir uma vigência nova basta acrescentar uma linha nesse script e rodá-lo, sem
mexer no notebook. Para as vigências cujo PCAT não traz o Desconto Social, a tarifa desse benefício é
derivada da residencial (coluna `ORIGEM`).

O PCAT escreve o posto por extenso e o BW usa o código abreviado do SAP; o casamento é feito
aplicando sobre a tabela a mesma regra do Passo 3.

O peso de cada vigência no mês é a fração de dias do mês que ela cobre. Como a ponderação é linear,
`TA = BE + CVA + FIN` continua exata. No mês do reajuste e no seguinte, esse peso de calendário é só
o ponto de partida: o Passo 7b o substitui por uma estimativa.

In [ ]:
CAMADAS = ["TA", "BE", "CVA", "FIN"]

# O PCAT escreve o posto por extenso (Ponta / Fora ponta / Intermediario); o BW usa o
# codigo abreviado do SAP: prefixo E-consumo/D-demanda + sufixo P/FP/INT, sufixo vazio
# para Nao se aplica (ver deriva_posto, Passo 3). Reproduz a mesma regra aqui, na direcao
# PCAT -> abreviado, para que a tabela de tarifas fique indexada no MESMO formato que
# bw["CHAVE"] usa - e o merge do Passo 5 continue um simples isin()/reindex().
SUFIXO_POSTO = {"Ponta": "P", "Fora ponta": "FP", "Intermediário": "INT", NAO_SE_APLICA: ""}

def _chave_pcat(row):
    prefixo = "D" if row["UNIDADE"] == "kW" else "E"
    posto_abrev = prefixo + SUFIXO_POSTO.get(row["POSTO"], "")
    # mesma excecao do deriva_posto (Passo 3): consumo sem ponta/fora ponta/intermediario
    # dessa subclasse especifica sai como "Nao se aplica" por extenso, nao como "E" solto.
    if posto_abrev == "E" and row["SUBCLASSE"] == "Residencial Desconto Social – faixa 02":
        posto_abrev = NAO_SE_APLICA
    return "".join([row["SUBGRUPO"], row["MODALIDADE"], row["CLASSE"], row["SUBCLASSE"],
                    row["DETALHE"], row["ACESSANTE"], posto_abrev, row["UNIDADE"]])

_long = pd.read_csv(ARQ_TARIFA_LONG, sep=";", decimal=",", encoding="utf-8-sig")
_long["VIGENCIA_INICIO"] = pd.to_datetime(_long["VIGENCIA_INICIO"]).dt.date
_long["VIGENCIA_FIM"]    = pd.to_datetime(_long["VIGENCIA_FIM"]).dt.date
_long["CHAVE"] = _long.apply(_chave_pcat, axis=1)

# Desconto Social: the rows come from the tariff table itself. For vintages whose
# PCAT predates the benefit, extrai_pcat.py derives them from the Residencial row
# (ORIGEM = "DERIVADA_RESIDENCIAL"). Tarifa Social (PCAT) is the Baixa Renda
# benefit, NOT the Desconto Social - there is no alias between the two.
print("linhas de tarifa derivadas (nao publicadas no PCAT):",
      int((_long["ORIGEM"] != "PCAT").sum()))

_NAO_COMPONENTE = {"ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM", "CAMADA", "CHAVE", "ORIGEM",
                   "SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE",
                   "POSTO", "UNIDADE", "ACESSANTE", "Total TUSD", "Total TE"}
COMPONENTES = [c for c in _long.columns if c not in _NAO_COMPONENTE]

# vigencias que tocam o mes de REFERENCIA, com o peso (dias de sobreposicao / dias do mes)
_mes_ini = dt.date(int(REFERENCIA[:4]), int(REFERENCIA[5:7]), 1)
_mes_fim = (pd.Timestamp(_mes_ini) + pd.offsets.MonthEnd(0)).date()
_dias_mes = (_mes_fim - _mes_ini).days + 1

_vigs = (_long[["ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM"]]
         .drop_duplicates().sort_values("VIGENCIA_INICIO"))
VIGENCIAS_NO_MES = []
for _, v in _vigs.iterrows():
    ini = max(v["VIGENCIA_INICIO"], _mes_ini)
    fim = min(v["VIGENCIA_FIM"], _mes_fim)
    dias = (fim - ini).days + 1
    if dias > 0:
        VIGENCIAS_NO_MES.append((int(v["ANO_VIGENCIA"]), dias / _dias_mes))

if not VIGENCIAS_NO_MES:
    raise ValueError(f"nenhuma vigencia em {ARQ_TARIFA_LONG.name} cobre o mes {REFERENCIA} "
                      "- atualize tbl/tarifas/tarifa_long.csv (3-Py/1-Entrega/extrai_pcat.py)")

print("vigencias em", REFERENCIA, ":",
      ", ".join(f"{ano} ({100*p:.1f}%)" for ano, p in VIGENCIAS_NO_MES))
print(f"{len(COMPONENTES)} componentes")

# subgroup of each tariff key, straight from the tariff table (no slicing of the key text)
SUBGRUPO_DA_CHAVE = _long.drop_duplicates("CHAVE").set_index("CHAVE")["SUBGRUPO"].to_dict()

# Candidate vintages: the ones touching the month plus the one in force on the first day of the
# previous month. Two candidates = reajuste month or the month after it (billing cycles still
# carry days of the old tariff). Only months with a single candidate feed the calibration.
def _vigencia_em(dia):
    for _, v in _vigs.iterrows():
        if v["VIGENCIA_INICIO"] <= dia <= v["VIGENCIA_FIM"]:
            return int(v["ANO_VIGENCIA"])
    return None

_ini_mes_ant = (pd.Timestamp(_mes_ini) - pd.offsets.MonthBegin(1)).date()
VIGENCIAS_CANDIDATAS = sorted({ano for ano, _ in VIGENCIAS_NO_MES} |
                              ({_vigencia_em(_ini_mes_ant)} - {None}))
print("vigencias candidatas (tarifa proporcional):", VIGENCIAS_CANDIDATAS)

PESO_ESTIMADO = {}   # filled by Passo 7b: {subgrupo: [(ano, peso), ...]}

def peso_da_vigencia(subgrupo):
    """Weights [(vintage, weight), ...] for one subgroup. Precedence: manual DIAS_POR_SUBGRUPO
    (only when there are exactly two candidate vintages), then the revenue-based estimate of
    Passo 7b, then the calendar."""
    if subgrupo in DIAS_POR_SUBGRUPO and len(VIGENCIAS_CANDIDATAS) == 2:
        ant, atu = DIAS_POR_SUBGRUPO[subgrupo]
        total = ant + atu
        if total == 0:
            return VIGENCIAS_NO_MES
        return [(VIGENCIAS_CANDIDATAS[0], ant / total), (VIGENCIAS_CANDIDATAS[1], atu / total)]
    if subgrupo in PESO_ESTIMADO:
        return PESO_ESTIMADO[subgrupo]
    return VIGENCIAS_NO_MES


In [ ]:
# Uma tabela de tarifa por camada, ja ponderada pelas vigencias do mes, indexada pela
# chave. Chave sem correspondencia numa das vigencias do mes usa so as vigencias em que
# existe, com o peso redistribuido entre elas; se nao existir em nenhuma, fica de fora.
_por_camada = {c: _long[_long["CAMADA"] == c].set_index(["ANO_VIGENCIA", "CHAVE"])[COMPONENTES]
               for c in CAMADAS}

todas_chaves = sorted(set(_long["CHAVE"]))

def monta_tarifa():
    """Weighted tariff per layer (dict camada -> DataFrame chave x componente), using
    peso_da_vigencia() per subgroup. Returns (tarifa, keys missing in some vintage)."""
    tarifa, faltando = {}, set()
    for camada in CAMADAS:
        base = _por_camada[camada]
        linhas, chaves_ok = [], []
        for chave in todas_chaves:
            pesos = [(a, p) for a, p in peso_da_vigencia(SUBGRUPO_DA_CHAVE[chave]) if p > 0]
            disponiveis = [(ano, p) for ano, p in pesos if (ano, chave) in base.index]
            if not disponiveis:
                # key absent from the estimated vintage: fall back to the calendar vintages
                disponiveis = [(ano, p) for ano, p in VIGENCIAS_NO_MES if (ano, chave) in base.index]
            if not disponiveis:
                continue
            if len(disponiveis) < len(pesos):
                faltando.add(chave)
            total_p = sum(p for _, p in disponiveis)
            valor = sum((p / total_p) * base.loc[(ano, chave)].astype(float).values
                        for ano, p in disponiveis)
            linhas.append(valor)
            chaves_ok.append(chave)
        tarifa[camada] = pd.DataFrame(linhas, index=chaves_ok, columns=COMPONENTES)
    return tarifa, faltando

TARIFA, sem_alguma_vigencia = monta_tarifa()

print(f"tarifa ponderada montada para {len(TARIFA['TA'])} chaves")
if sem_alguma_vigencia:
    print(f"ATENCAO: {len(sem_alguma_vigencia)} chaves sem correspondencia em alguma "
          "vigencia do mes; peso redistribuido entre as vigencias em que a chave existe")

# conferencia: TA = BE + CVA + FIN, componente a componente
maior = (TARIFA["TA"] - TARIFA["BE"] - TARIFA["CVA"] - TARIFA["FIN"]).abs().max().max()
print(f"TA = BE + CVA + FIN, diferenca maxima: {maior:.10f}")


---
## Passo 5 — A quantidade

O consumo é convertido de kWh para MWh, porque a tarifa de energia é publicada em R$/MWh; a demanda
fica em kW.

**O SAP lança a quantidade uma vez só.** No cliente cativo, o volume aparece apenas na linha lançada
na parcela TE, e a receita é dividida entre essa linha e a da TUSD, que vem com quantidade zero. No
cliente livre é o espelho: toda a quantidade está do lado TUSD, porque a distribuidora não fatura a
energia dele. Por isso o cálculo é sempre lido pelo par TUSD/TE, nunca pela linha isolada.

In [ ]:
bw["CONSUMO_MWH"] = (bw["QTD_CONSUMO_TUSD"] + bw["QTD_CONSUMO_TE"]) / 1000.0
bw["DEMANDA_KW"]  = bw["QTD_DEMANDA_TUSD"] + bw["QTD_DEMANDA"]
bw["QUANTIDADE"]  = np.where(bw["GRANDEZA"] == "CONSUMO", bw["CONSUMO_MWH"], bw["DEMANDA_KW"])

# conferencia contra as colunas de controle do proprio arquivo
d1 = (bw["CONSUMO_MWH"] * 1000 - bw["TOTAL_CONSUMO_XLS"]).abs().max()
d2 = (bw["DEMANDA_KW"] - bw["TOTAL_DEMANDA_XLS"]).abs().max()
print(f"consumo calculado x total de consumo da base : diferenca maxima {d1:.6f}")
print(f"demanda calculada x total de demanda da base : diferenca maxima {d2:.6f}")

---
## Passo 6 — Quais parcelas incidem

| Mercado | Grandeza | Linha lançada em | Aplica TUSD | Aplica TE |
|---|---|---|:-:|:-:|
| Cativo | Consumo | TE | sim | sim |
| Cativo | Consumo | TUSD | sim | não |
| Cativo | Demanda | TUSD | sim | não |
| Livre | Consumo e demanda | TUSD | sim | não |
| Híbrido | Consumo e demanda | qualquer | sim | não |

O cativo paga TUSD **e** TE sobre o mesmo volume, e é na linha lançada em TE que o volume está — por
isso ela recebe as duas parcelas. A linha lançada em TUSD tem quantidade zero e não recebe nada; se
recebesse, a energia seria contada duas vezes. O livre compra a energia no mercado e paga só a TUSD;
o híbrido fatura como livre.

In [ ]:
# (mercado, grandeza, parcela do lancamento) -> parcelas da tarifa que incidem
REGRA_VALORIZACAO = {
    ("Cliente Cativo",  "CONSUMO", "TE"):   ("TUSD", "TE"),
    ("Cliente Cativo",  "CONSUMO", "TUSD"): ("TUSD",),
    ("Cliente Cativo",  "DEMANDA", "TUSD"): ("TUSD",),
    ("Cliente Livre",   "CONSUMO", "TUSD"): ("TUSD",),
    ("Cliente Livre",   "DEMANDA", "TUSD"): ("TUSD",),
    ("Cliente Híbrido", "CONSUMO", "TE"):   ("TUSD",),
    ("Cliente Híbrido", "CONSUMO", "TUSD"): ("TUSD",),
    ("Cliente Híbrido", "DEMANDA", "TUSD"): ("TUSD",),
}

# cada componente pertence a uma parcela e a uma familia
def parcela_do_componente(nome):
    return "TE" if nome.startswith("TE") else "TUSD"

FAMILIA = {
    "FIO A":   ["TUSD_RB", "TUSD_FR", "TUSD_CCT", "TUSD_CCD", "TUSD_CUSD", "TUSDG_T",
                "TE_TRANSPORTE_ITAIPU", "TE_TUST_ITAIPU", "TE_TUST_CI"],
    "FIO B":   ["TUSD_FioB"],
    "ENCARGOS":["TUSD_CDE_COVID", "TUSD_TFSEE", "TUSD_PeD", "TUSD_ONS", "TUSD_CCC", "TUSD_CDE",
                "TUSD_PROINFA", "Liminar1", "TUSDG_ONS", "TE_CDE_COVID", "TE_CDE_ELET",
                "TE_CDE_GD", "TE_PeD", "TE_ESSERR", "TE_CFURH"],
    "PERDAS":  ["TUSD_PT", "TUSD_Per_RB_D", "TUSD_PNT", "TE_Per_RB"],
    "ENERGIA": ["TE_ENERGIA", "TE_ANGRA"],
    "RI":      ["TUSD_RI"],
    "OUTROS":  ["TUSD Subsidio", "TUSD BENEFICIO_L14299", "TUSD Outros",
                "TE Subsidio", "TE BENEFICIO_L14299"],
}
DE_PARA_FAMILIA = {c: f for f, lista in FAMILIA.items() for c in lista}

faltando = [c for c in COMPONENTES if c not in DE_PARA_FAMILIA]
assert not faltando, f"componentes sem familia: {faltando}"
print(f"{len(COMPONENTES)} componentes classificados em {len(FAMILIA)} familias")

---
## Passo 7 — Multiplicar: o bloco 1

Para cada linha tarifável, cada um dos 37 itens e cada uma das 4 camadas:

```
valor do item = quantidade × tarifa do item     (zero se a parcela do item não incide na linha)
```

Nas subclasses Baixa Renda e Desconto Social, os **itens isentos de encargos** recebem tarifa zero
(PRORET Submódulo 7.1, §18 e §23): Quota CDE, PROINFA e CDE COVID na TUSD e CDE GD na TE para o Baixa
Renda; os mesmos, sem o PROINFA, para o Desconto Social. Não é um desconto percentual: é a não
incidência desses itens.

> O valor calculado é a **soma dos 37 itens**. A tarifa total publicada vem arredondada em duas
> casas, por isso pode diferir alguns milhares de reais no mês de quem calcule pela tarifa total.

In [ ]:
tarifaveis = bw[bw["TARIFAVEL"] & bw["CHAVE"].isin(TARIFA["TA"].index)].copy()
orfas = bw[bw["TARIFAVEL"] & ~bw["CHAVE"].isin(TARIFA["TA"].index)]
if len(orfas):
    print(f"ATENCAO: {len(orfas)} linha(s) tarifavel(is) com chave sem tarifa na tabela de tarifas:")
    for _, r in orfas.iterrows():
        print(f"   linha {r['LINHA_EXCEL']} | {r['GRP_QUANT']} | {r['CHAVE']}")

# mascara por linha: o componente incide ou nao
parcelas_aplicadas = tarifaveis.apply(
    lambda r: REGRA_VALORIZACAO.get(
        (r["FLAG_MERCADO"], r["GRANDEZA"], r["PARCELA_LANCAMENTO"]), ()), axis=1)
aplica = pd.DataFrame(
    {c: parcelas_aplicadas.apply(lambda p, c=c: parcela_do_componente(c) in p)
     for c in COMPONENTES}, index=tarifaveis.index)

sem_regra = parcelas_aplicadas.apply(len) == 0
if sem_regra.any():
    print(f"ATENCAO: {sem_regra.sum()} linha(s) sem regra de valorizacao")

# isencao de encargos setoriais para baixa renda / desconto social (ver Passo 3)
tarifaveis["BENEFICIO_SOCIAL"] = tarifaveis["CLASSE_SUBCLASSE"].map(SUBCLASSE_BENEFICIO)
isento = pd.DataFrame(
    {c: tarifaveis["BENEFICIO_SOCIAL"].map(
            lambda b, c=c: c in ITENS_ISENTOS_BENEFICIO.get(b, ()))
     for c in COMPONENTES}, index=tarifaveis.index)
aplica_regra = aplica.copy()          # rule mask, before the social exemption
aplica = aplica & ~isento

n_beneficio = (tarifaveis["BENEFICIO_SOCIAL"].notna()).sum()
if n_beneficio:
    print(f"isencao de encargos aplicada em {n_beneficio} linha(s) "
          f"({tarifaveis.loc[tarifaveis['BENEFICIO_SOCIAL'].notna(), 'BENEFICIO_SOCIAL'].value_counts().to_dict()})")

def valoriza_bloco1(tarifa):
    """Bloco 1 per layer: quantity x tariff, component by component, masked by the rule."""
    bloco = {}
    for camada in CAMADAS:
        tab = tarifa[camada].reindex(tarifaveis["CHAVE"].values)
        tab.index = tarifaveis.index
        bloco[camada] = tab[COMPONENTES].mul(tarifaveis["QUANTIDADE"], axis=0).where(aplica, 0.0)
    return bloco

BLOCO1 = valoriza_bloco1(TARIFA)

tarifaveis["CALCULADO_TA"] = BLOCO1["TA"].sum(axis=1)
print(f"\nvalor calculado pela tarifa de aplicacao: R$ {tarifaveis['CALCULADO_TA'].sum():,.2f}")
print(f"receita registrada nessas linhas        : R$ {tarifaveis['RECEITA_BRL'].sum():,.2f}")

---
## Passo 7b — Tarifa proporcional nos meses de reajuste

Quando a tarifa muda no decorrer do ciclo de faturamento, a distribuidora aplica a tarifa
proporcional aos dias de cada vigência dentro do ciclo de leitura (REN ANEEL 1.000/2021, art. 173,
§ 2º). Como a extração do BW não traz a data de leitura, o peso de cada vigência não é o do calendário
do mês: depende do calendário de leitura de cada subgrupo. Estimo esse peso pela própria receita,
só na base **cativa sem benefício social**, em que a receita é quantidade × tarifa sem desconto:

1. **Histórico.** Todo mês processado grava `tbl/calibracao/calibracao_<ref>.csv` com receita e valor
   calculado por subgrupo × classe. Nos meses com uma vigência só, a razão `k = receita / calculado`
   é a régua do subgrupo: mede o que a tarifa não explica (arredondamentos, acertos de faturamento).
   `k` = Σ receita / Σ calculado dos últimos `N_MESES_CALIBRACAO` meses válidos.
2. **Estimativa.** No mês com duas vigências candidatas, `A` é o valor do subgrupo 100% na vigência
   antiga e `B` 100% na nova. O peso da nova é `p = (receita / k − A) / (B − A)`, limitado a [0, 1].
3. **Salvaguardas.** Fica com o peso do calendário o subgrupo sem histórico, com reajuste menor que
   `LIMIAR_DIF_TARIFA` (a conta divide por `B − A`, que fica instável) ou com receita cativa abaixo de
   `MIN_RECEITA_CALIBRACAO`. `DIAS_POR_SUBGRUPO`, se preenchido, prevalece.
4. **Aplicação.** O peso estimado vale para todas as linhas do subgrupo (cativo, livre, benefício
   social e SD), e a tarifa e o bloco 1 são refeitos.
5. **Conferência.** Por classe, visão independente da estimativa (feita por subgrupo): o erro
   `receita / (k_classe × calculado) − 1` com o peso do calendário e com o peso estimado. A receita do
   BW é a mesma do informe de gestão, então o erro geral é o erro contra o informe.

In [ ]:
MES_DE_REAJUSTE = len(VIGENCIAS_CANDIDATAS) == 2
VALIDO_CALIBRACAO = len(VIGENCIAS_CANDIDATAS) == 1
base_cal = (tarifaveis["FLAG_MERCADO"] == "Cliente Cativo") & tarifaveis["BENEFICIO_SOCIAL"].isna()

def valor_puro(ano):
    """TA value of each line at 100% of one vintage; NaN where the key does not exist there."""
    tab = _por_camada["TA"].xs(ano, level="ANO_VIGENCIA").astype(float)
    tab = tab.reindex(tarifaveis["CHAVE"].values)
    tab.index = tarifaveis.index
    valor = tab[COMPONENTES].mul(tarifaveis["QUANTIDADE"], axis=0).where(aplica, 0.0).sum(axis=1)
    return valor.where(tab[COMPONENTES].notna().any(axis=1))

def carrega_historico():
    """Valid calibration months before REFERENCIA, last N_MESES_CALIBRACAO of them."""
    arquivos = sorted(DIR_CALIBRACAO.glob("calibracao_*.csv"))
    hist = [pd.read_csv(a, sep=";", decimal=",", encoding="utf-8-sig") for a in arquivos]
    hist = pd.concat(hist, ignore_index=True) if hist else pd.DataFrame(
        columns=["REFERENCIA", "SUBGRUPO", "CLASSE_PRINCIPAL", "RECEITA_BRL", "CALCULADO_TA",
                 "VALIDO_CALIBRACAO"])
    hist = hist[(hist["REFERENCIA"].astype(str) < REFERENCIA) &
                (hist["VALIDO_CALIBRACAO"].astype(str).str.lower() == "true")]
    refs = sorted(hist["REFERENCIA"].astype(str).unique())[-N_MESES_CALIBRACAO:]
    return hist[hist["REFERENCIA"].astype(str).isin(refs)], refs

def razao(df, por):
    g = df.groupby(por)[["RECEITA_BRL", "CALCULADO_TA"]].sum()
    return (g["RECEITA_BRL"] / g["CALCULADO_TA"]).replace([np.inf, -np.inf], np.nan)

RELATORIO_CALIBRACAO = None
CONFERENCIA_CALIBRACAO = None
if not (1 <= N_MESES_CALIBRACAO <= 12):
    raise ValueError("N_MESES_CALIBRACAO deve ficar entre 1 e 12")

if CALIBRAR_TARIFA_PROPORCIONAL and MES_DE_REAJUSTE:
    hist, refs_usadas = carrega_historico()
    print(f"historico de calibracao: {len(refs_usadas)} mes(es) {refs_usadas} "
          f"(pedido: {N_MESES_CALIBRACAO})")
    k_sub = razao(hist, "SUBGRUPO")
    k_cla = razao(hist, "CLASSE_PRINCIPAL")
    ano_ant, ano_nov = VIGENCIAS_CANDIDATAS
    A, B = valor_puro(ano_ant), valor_puro(ano_nov)
    ok = base_cal & A.notna() & B.notna()
    df = tarifaveis.loc[ok, ["SUBGRUPO", "CLASSE_PRINCIPAL", "RECEITA_BRL", "CALCULADO_TA"]].assign(
        A=A[ok], B=B[ok])
    peso_cal = dict(VIGENCIAS_NO_MES).get(ano_nov, 0.0)

    relatorio = []
    for s, r in df.groupby("SUBGRUPO")[["RECEITA_BRL", "A", "B"]].sum().iterrows():
        k = k_sub.get(s, np.nan)
        dif = (r["B"] - r["A"]) / r["A"] if r["A"] else np.nan
        p_bruto, p = np.nan, peso_cal
        if s in DIAS_POR_SUBGRUPO:
            regra = "manual (DIAS_POR_SUBGRUPO)"
            p = dict(peso_da_vigencia(s)).get(ano_nov, 0.0)
        elif np.isnan(k):
            regra = "calendario - sem historico"
        elif np.isnan(dif) or abs(dif) < LIMIAR_DIF_TARIFA:
            regra = "calendario - reajuste pequeno"
        elif r["RECEITA_BRL"] < MIN_RECEITA_CALIBRACAO:
            regra = "calendario - receita pequena"
        else:
            p_bruto = (r["RECEITA_BRL"] / k - r["A"]) / (r["B"] - r["A"])
            p = float(np.clip(p_bruto, 0.0, 1.0))
            regra = "estimado" if p == p_bruto else "estimado (limitado a 0-100%)"
            PESO_ESTIMADO[s] = [(ano_ant, 1.0 - p), (ano_nov, p)]
        relatorio.append({"SUBGRUPO": s, "RECEITA_BRL": r["RECEITA_BRL"], "K_HISTORICO": k,
                          "VALOR_100_ANTIGA": r["A"], "VALOR_100_NOVA": r["B"], "REAJUSTE_PCT": dif,
                          "PESO_NOVA_CALENDARIO": peso_cal, "PESO_NOVA_ESTIMADO_BRUTO": p_bruto,
                          "PESO_NOVA_USADO": p, "REGRA": regra})
    RELATORIO_CALIBRACAO = pd.DataFrame(relatorio)

    # rebuild tariff and bloco 1 with the estimated weights
    calc_calendario = tarifaveis["CALCULADO_TA"].copy()
    TARIFA, sem_alguma_vigencia = monta_tarifa()
    BLOCO1 = valoriza_bloco1(TARIFA)
    tarifaveis["CALCULADO_TA"] = BLOCO1["TA"].sum(axis=1)

    # check by class (class is not used in the estimate): error vs the class historical ratio
    conf = tarifaveis.loc[base_cal].assign(CAL_CALENDARIO=calc_calendario[base_cal])
    conf = conf.groupby("CLASSE_PRINCIPAL")[["RECEITA_BRL", "CAL_CALENDARIO", "CALCULADO_TA"]].sum()
    conf.loc["TOTAL"] = conf.sum()
    kc = k_cla.reindex(conf.index)
    kc["TOTAL"] = (hist["RECEITA_BRL"].sum() / hist["CALCULADO_TA"].sum()) if len(hist) else np.nan
    conf["K_HISTORICO"] = kc
    conf["ERRO_CALENDARIO"] = conf["RECEITA_BRL"] / (conf["K_HISTORICO"] * conf["CAL_CALENDARIO"]) - 1
    conf["ERRO_ESTIMADO"] = conf["RECEITA_BRL"] / (conf["K_HISTORICO"] * conf["CALCULADO_TA"]) - 1
    CONFERENCIA_CALIBRACAO = conf.reset_index()

    pd.set_option("display.width", 200)
    print(RELATORIO_CALIBRACAO[["SUBGRUPO", "RECEITA_BRL", "K_HISTORICO", "REAJUSTE_PCT",
                                "PESO_NOVA_CALENDARIO", "PESO_NOVA_USADO", "REGRA"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
    print()
    print(CONFERENCIA_CALIBRACAO[["CLASSE_PRINCIPAL", "RECEITA_BRL", "ERRO_CALENDARIO", "ERRO_ESTIMADO"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
elif MES_DE_REAJUSTE:
    print("mes com duas vigencias candidatas; calibracao desligada - peso do calendario")
else:
    print("mes com uma vigencia so: sem estimativa; mes entra no historico de calibracao")

# monthly calibration file (written every run; only single-vintage months are valid)
DIR_CALIBRACAO.mkdir(parents=True, exist_ok=True)
pesos_txt = {s: " | ".join(f"{a}:{p:.4f}" for a, p in peso_da_vigencia(s))
             for s in tarifaveis["SUBGRUPO"].unique()}
cal_mes = (tarifaveis.loc[base_cal]
           .groupby(["SUBGRUPO", "CLASSE_PRINCIPAL"])
           .agg(RECEITA_BRL=("RECEITA_BRL", "sum"), CALCULADO_TA=("CALCULADO_TA", "sum"),
                N_LINHAS=("RECEITA_BRL", "size"))
           .reset_index())
cal_mes.insert(0, "REFERENCIA", REFERENCIA)
cal_mes["PESOS_VIGENCIA"] = cal_mes["SUBGRUPO"].map(pesos_txt)
cal_mes["VALIDO_CALIBRACAO"] = VALIDO_CALIBRACAO
ARQ_CALIBRACAO = DIR_CALIBRACAO / f"calibracao_{REFERENCIA}.csv"
cal_mes.to_csv(ARQ_CALIBRACAO, sep=";", decimal=",", index=False, encoding="utf-8-sig")
print(f"\ngravado: {ARQ_CALIBRACAO.name} | valido para calibracao: {VALIDO_CALIBRACAO} | "
      f"razao receita/calculado do mes: {cal_mes['RECEITA_BRL'].sum() / cal_mes['CALCULADO_TA'].sum():.4f}")


---
## Passo 8 — Agrupar e estimar as faixas dos benefícios sociais

**Agrupamento.** A saída traz uma linha por **grupo**, não por linha do BW. O grupo junta os
lançamentos que descrevem o mesmo fornecimento:

```
Classe principal + Categoria tarifa + Operação principal + Flag mercado + Classe/Sub-classe + Chave
+ Tipo de faturamento
```

Os campos que variam dentro do par TUSD/TE são concatenados na mesma célula (por exemplo,
`ZQC153 | ZQC154`), e a coluna `Linhas de origem` guarda as linhas da extração que formam cada grupo.
Sem isso, metade das linhas mostraria só a parcela TUSD e a outra metade a TE. Como o tipo de
faturamento faz parte do grupo, FAT, REFAT e CNR passam pelo bloco 2 separados e cada um fecha na
própria receita.

**Faixas dos benefícios sociais.** A Tarifa Social isenta o consumo até 80 kWh/mês e o Desconto Social
aplica tarifa própria até 120 kWh/mês (REN ANEEL 1.000/2021, art. 179, § 1º e § 4º). O BW não traz o
consumo por cliente, então a participação de cada faixa é estimada pela receita do grupo:

- `A` = valor se toda a energia estivesse na faixa 1; `B` = valor se toda estivesse na faixa 2;
- participação da faixa 2 = `(receita − A) / (B − A)`, limitada a [0, 1];
- o bloco 1 do grupo passa a ser `A + participação × (B − A)`, item a item e camada a camada.

Na Tarifa Social, `A` = 0 (faixa isenta) e `B` é a tarifa baixa renda com a isenção de encargos. No
Desconto Social, `A` é a tarifa do Desconto Social e `B` a residencial comum. A participação estimada
aparece na coluna `% energia na faixa 01`.

In [ ]:
CHAVES_DO_GRUPO = ["CLASSE_PRINCIPAL", "CATEGORIA_TARIFA", "OPERACAO", "OPERACAO_TXT",
                   "FLAG_MERCADO", "CLASSE_SUBCLASSE", "CHAVE", "TIPO_FATURAMENTO"]
CONCATENADAS = ["GRP_MONT", "GRP_QUANT", "GRP_QUANT_TXT", "SUBOPERACAO", "SUBOPERACAO_TXT"]
QUANTIDADES = {"CONSUMO_TUSD": "QTD_CONSUMO_TUSD", "CONSUMO_TE": "QTD_CONSUMO_TE",
               "DEMANDA": "DEMANDA_KW"}

def junta(serie, limite=6):
    """Valores distintos, na ordem em que aparecem, separados por barra."""
    vistos, saida = set(), []
    for v in serie:
        t = str(v).strip()
        if t and t.lower() != "nan" and t not in vistos:
            vistos.add(t); saida.append(t)
    if len(saida) > limite:
        return " | ".join(saida[:limite]) + f" | ... (+{len(saida) - limite})"
    return " | ".join(saida)

def lista_linhas(serie, limite=10):
    """Numeros das linhas da extracao que compoem o grupo."""
    numeros = sorted(int(v) for v in serie if pd.notna(v))   # SD non-tariff lines have no BW line
    if len(numeros) > limite:
        return "-".join(str(n) for n in numeros[:limite]) + f"-... (+{len(numeros) - limite})"
    return "-".join(str(n) for n in numeros)

grupo = tarifaveis.groupby(CHAVES_DO_GRUPO, dropna=False, sort=False)
ident = grupo.agg(**{c: (c, junta) for c in CONCATENADAS})
ident["LINHAS_ORIGEM"] = grupo["LINHA_EXCEL"].agg(lista_linhas)
for nome, coluna in QUANTIDADES.items():
    ident[nome] = grupo[coluna].sum()
ident["TOTAL_CONSUMO"] = grupo["TOTAL_CONSUMO_XLS"].sum()
ident["TOTAL_DEMANDA"] = grupo["TOTAL_DEMANDA_XLS"].sum()
ident["RECEITA_BRL"]   = grupo["RECEITA_BRL"].sum()
ident["CALCULADO_TA"]  = grupo["CALCULADO_TA"].sum()
ident = ident.reset_index()

# os blocos de valor somados pelo mesmo agrupamento
codigo_grupo = grupo.ngroup()
B1_GRUPO = {c: BLOCO1[c].groupby(codigo_grupo).sum().reset_index(drop=True) for c in CAMADAS}

print(f"{len(tarifaveis)} linhas tarifaveis do BW -> {len(ident)} grupos")
# --- Faixas de consumo dos beneficios sociais, estimadas pela receita (PROCEDIMENTO_PXQ.md, Passo 8)
# O BW nao traz a faixa de consumo por UC. Cada grupo de beneficio e valorizado nas
# duas faixas (A = faixa 01, B = faixa 02) e a parcela de energia de cada faixa e a
# que reproduz a receita faturada do grupo: s2 = (R - A) / (B - A), limitada a [0, 1].
#   Baixa Renda (Tarifa Social): faixa 01 = ate 80 kWh, reducao de 100% -> A = 0;
#                                faixa 02 = tarifa Baixa Renda com a isencao de 4.5.
#   Desconto Social:             faixa 01 = ate 120 kWh, chave "... faixa 01";
#                                faixa 02 = chave "... faixa 02" (Residencial), sem isencao.
DS_F1 = "Residencial Desconto Social – faixa 01"
DS_F2 = "Residencial Desconto Social – faixa 02"

def chave_ds_faixa01(r):
    """Same key layout as monta_chave, pointing to the faixa 01 subclass."""
    return "".join([r["SUBGRUPO"], r["MODALIDADE"], r["CLASSE"], DS_F1, r["DETALHE"],
                    r["UC"], "E", r["UNIDADE"]])

beneficio = tarifaveis["BENEFICIO_SOCIAL"]
linha_br = beneficio == "BAIXA_RENDA"
linha_ds = ((beneficio == "DESCONTO_SOCIAL") & (tarifaveis["SUBCLASSE"] == DS_F2)
            & (tarifaveis["GRANDEZA"] == "CONSUMO"))
# months without Desconto Social rows (benefit started mid-2026 in the BW) -> empty Series;
# DataFrame.apply on an empty frame would return a DataFrame instead of a Series
chave_f1 = (tarifaveis.loc[linha_ds].apply(chave_ds_faixa01, axis=1) if linha_ds.any()
            else pd.Series(dtype=object))
sem_f1 = ~chave_f1.isin(TARIFA["TA"].index)
if sem_f1.any():
    print(f"ATENCAO: {int(sem_f1.sum())} linha(s) de Desconto Social sem tarifa de faixa 01 - "
          "ficam sem estimativa de faixa")
    chave_f1 = chave_f1[~sem_f1]
idx_ds = chave_f1.index

VALOR_A, VALOR_B = {}, {}
for camada in CAMADAS:
    a = BLOCO1[camada].copy()
    b = BLOCO1[camada].copy()
    a.loc[linha_br] = 0.0
    if len(idx_ds):
        q = tarifaveis.loc[idx_ds, "QUANTIDADE"]
        t1 = TARIFA[camada].reindex(chave_f1.values)
        t1.index = idx_ds
        a.loc[idx_ds] = t1[COMPONENTES].mul(q, axis=0).where(aplica_regra.loc[idx_ds], 0.0)
        t2 = TARIFA[camada].reindex(tarifaveis.loc[idx_ds, "CHAVE"].values)
        t2.index = idx_ds
        b.loc[idx_ds] = t2[COMPONENTES].mul(q, axis=0).where(aplica_regra.loc[idx_ds], 0.0)
    VALOR_A[camada], VALOR_B[camada] = a, b

em_faixa = linha_br | tarifaveis.index.isin(idx_ds)
grupo_faixa = pd.Series(em_faixa, index=tarifaveis.index).groupby(codigo_grupo).any()
a_g = VALOR_A["TA"].sum(axis=1).groupby(codigo_grupo).sum()
b_g = VALOR_B["TA"].sum(axis=1).groupby(codigo_grupo).sum()
r_g = tarifaveis["RECEITA_BRL"].groupby(codigo_grupo).sum()
s2_g = ((r_g - a_g) / (b_g - a_g).replace(0, np.nan)).clip(0.0, 1.0).fillna(1.0)
s2_g = s2_g.where(grupo_faixa, 1.0)
s2_linha = codigo_grupo.map(s2_g)

antes_ta = tarifaveis["CALCULADO_TA"].sum()
for camada in CAMADAS:
    BLOCO1[camada] = VALOR_A[camada] + (VALOR_B[camada] - VALOR_A[camada]).mul(s2_linha, axis=0)
tarifaveis["CALCULADO_TA"] = BLOCO1["TA"].sum(axis=1)
B1_GRUPO = {c: BLOCO1[c].groupby(codigo_grupo).sum().reset_index(drop=True) for c in CAMADAS}
ident["CALCULADO_TA"] = B1_GRUPO["TA"].sum(axis=1).values
ident["FAIXA1_PCT"] = np.where(grupo_faixa.values, 1.0 - s2_g.values, np.nan)

print("\nfaixas dos beneficios sociais estimadas pela receita:")
for nome, marca in (("Baixa Renda (ate 80 kWh)", linha_br),
                    ("Desconto Social (ate 120 kWh)", tarifaveis.index.isin(idx_ds))):
    gs = sorted(set(codigo_grupo[marca]))
    if not gs:
        continue
    e = tarifaveis.loc[marca, "QUANTIDADE"].groupby(codigo_grupo[marca]).sum()
    pct = (e * (1.0 - s2_g.loc[e.index])).sum() / e.sum()
    travado = int(((s2_g.loc[gs] <= 0.0) | (s2_g.loc[gs] >= 1.0)).sum())
    print(f"  {nome}: {len(gs)} grupo(s), {100 * pct:.1f}% da energia na faixa 01"
          + (f" | ATENCAO: {travado} grupo(s) no limite 0%/100%" if travado else ""))
print(f"  bloco 1 antes da estimativa R$ {antes_ta:,.2f} | depois R$ {tarifaveis['CALCULADO_TA'].sum():,.2f}")

ident[["CATEGORIA_TARIFA", "GRP_QUANT", "LINHAS_ORIGEM", "RECEITA_BRL", "CALCULADO_TA"]].head(4)

---
## Passo 9 — O ajuste: o bloco 2

A diferença entre o bloco 1 e a receita é corrigida em três etapas, nesta ordem. Cada etapa trata
uma causa e atua só sobre os itens que essa causa afeta.

**1. Regime APE** (parâmetro `REGIME_APE`). O autoprodutor por equiparação é isento dos encargos da
TUSD de energia. O BW não identifica o APE: a energia dele é faturada pela tarifa do livre e a
isenção vem em linhas próprias, sem quantidade — `TUSD - Desconto Consumo` (ZQC036 na Azul; ZQC181
ponta e ZQC182 fora ponta na Verde) e o estorno `COMPLEMENTO RES.ANEEL 166/05` (ZQC059 na Azul;
ZQC183 e ZQC184 na Verde), que acerta energia de meses anteriores.

- O valor líquido do mês vai para os grupos de energia TUSD dos livres do mesmo subgrupo, modalidade
  e posto (da mesma classe, quando houver), na proporção da **capacidade de isenção** de cada grupo:
  quantidade × (TUSD do livre − TUSD do APE).
- O valor abate só os itens da TUSD em que as duas tarifas diferem — na prática, CDE e PROINFA —, em
  cada camada.
- O que passar da capacidade de um grupo vai para os demais grupos do mesmo subgrupo e modalidade; se
  ainda sobrar, segue para as passadas seguintes e é informado.

**2. 1ª passada — desconto de fonte incentivada.** Clientes livres do grupo A (regime `fio`). O que
ainda separa o bloco 1 da receita é absorvido **somente por Fio A e Fio B**, com desconto limitado a
[0%, 100%]. Encargos e perdas ficam intactos (PRORET Submódulo 7.1, §38: o desconto incide sobre a
TUSD Transporte).

**3. 2ª passada — fechamento.** Todos os grupos. O que sobra é distribuído proporcionalmente entre
todos os itens do grupo, até o total igualar a receita. No cativo (regime `uniforme`), é o único
ajuste. A 2ª passada **não é um desconto** — por isso o tamanho dela é o indicador de qualidade do
processo. A saída mostra, por grupo, o ajuste da 1ª e o da 2ª passada.

Como todas as operações são multiplicações aplicadas igualmente às quatro camadas, `TA = BE + CVA +
FIN` continua exata.

**Parâmetro `APLICAR_2A_PASSADA`.** Com `False`, a 2ª passada não é feita e a sobra sai nas abas
`Diferencas_linha` (uma linha por linha da extração, com o número da linha no Excel e o grupo) e
`Diferencas_grupo`. Na leitura linha a linha, lembre que no cativo só o par TUSD/TE do mesmo grupo
fecha (Passo 5).

In [ ]:
# Componentes that carry the incentivized-source discount: Fio A and Fio B only.
COLS_FIO = [c for c in FAMILIA["FIO A"] + FAMILIA["FIO B"] if c in B1_GRUPO["TA"].columns]

# Adjustment regime per group.
SUBGRUPO_GRUPO = ident["CHAVE"].str.extract(PADRAO_SUBGRUPO, expand=False)
ident["REGIME"] = np.where(
    (ident["FLAG_MERCADO"] == "Cliente Livre") & SUBGRUPO_GRUPO.str.startswith("A").fillna(False),
    "fio", "uniforme")
em_fio = ident["REGIME"] == "fio"

# --- APE regime (REGIME_APE, Passo 0): exemption lines of the self-producer by equivalence
# enter the base of the free-market groups and reduce only the TUSD energy items where the
# APE tariff differs from the free-market tariff (CDE, PROINFA, ...).
APE_CODIGOS = {"ZQC036": ("EP", "EFP"), "ZQC059": ("EP", "EFP"),
               "ZQC181": ("EP",), "ZQC183": ("EP",), "ZQC182": ("EFP",), "ZQC184": ("EFP",)}
COMP_TUSD = [c for c in B1_GRUPO["TA"].columns if c.startswith("TUSD")]
APE_RED = {c: pd.DataFrame(0.0, index=ident.index, columns=B1_GRUPO[c].columns) for c in CAMADAS}
ident["RECEITA_APE_BRL"] = 0.0
ident["AJUSTE_APE_BRL"] = 0.0
APE_USADAS = pd.Index([])
APE_LINHA_GRUPO = {}          # BW line -> list of (group, value allocated)
if REGIME_APE:
    ape_linhas = bw[bw["GRP_QUANT"].astype(str).isin(APE_CODIGOS) & ~bw.index.isin(tarifaveis.index)]
    attr = tarifaveis.groupby(codigo_grupo)[["SUBGRUPO", "MODALIDADE", "POSTO", "UNIDADE",
                                             "FLAG_MERCADO", "PARCELA_LANCAMENTO"]].first()
    attr.index = ident.index
    qtd_g = tarifaveis.groupby(codigo_grupo)["QUANTIDADE"].sum()
    qtd_g.index = ident.index
    alvo = ((attr["FLAG_MERCADO"] == "Cliente Livre") & (attr["UNIDADE"] == "MWh")
            & (attr["PARCELA_LANCAMENTO"] == "TUSD") & attr["SUBGRUPO"].str.startswith("A"))

    # tariff difference free-market - APE, TUSD items only, per group and layer
    DIF_APE = {c: pd.DataFrame(0.0, index=ident.index, columns=COMP_TUSD) for c in CAMADAS}
    for g in ident.index[alvo]:
        a = attr.loc[g]
        k_livre, k_ape = (a["SUBGRUPO"] + a["MODALIDADE"] + NAO_SE_APLICA * 2 + det + NAO_SE_APLICA
                          + a["POSTO"] + "MWh" for det in (NAO_SE_APLICA, "APE"))
        if k_livre in TARIFA["TA"].index and k_ape in TARIFA["TA"].index:
            for c in CAMADAS:
                DIF_APE[c].loc[g] = TARIFA[c].loc[k_livre, COMP_TUSD] - TARIFA[c].loc[k_ape, COMP_TUSD]
    cap_g = qtd_g * DIF_APE["TA"].sum(axis=1)            # exemption capacity in R$
    cap_g = cap_g.where(alvo & (cap_g > 0), 0.0)

    sem_destino = []
    for i, r in ape_linhas.iterrows():
        cand = (cap_g > 0) & (attr["SUBGRUPO"] == r["SUBGRUPO"]) & (attr["MODALIDADE"] == r["MODALIDADE"]) \
               & attr["POSTO"].isin(APE_CODIGOS[str(r["GRP_QUANT"])])
        mesma = cand & (ident["CLASSE_SUBCLASSE"] == r["CLASSE_SUBCLASSE"])
        dest = mesma if mesma.any() else cand
        if not dest.any():
            sem_destino.append(i)
            continue
        peso = cap_g[dest] / cap_g[dest].sum()
        APE_LINHA_GRUPO[i] = list((peso * r["RECEITA_BRL"]).items())
        ident.loc[peso.index, "RECEITA_APE_BRL"] += (peso * r["RECEITA_BRL"]).values
    APE_USADAS = pd.Index(list(APE_LINHA_GRUPO))

    # discount above a group's capacity (credits from previous months, larger than this month's
    # energy) is moved to the other groups of the same subgroup and modality with spare capacity
    sobra = (-ident["RECEITA_APE_BRL"] - cap_g).clip(lower=0.0)
    for (sg, md), idx in attr[sobra > 0.01].groupby(["SUBGRUPO", "MODALIDADE"]).groups.items():
        livre_cap = (cap_g + ident["RECEITA_APE_BRL"]).clip(lower=0.0)
        viz = (cap_g > 0) & (attr["SUBGRUPO"] == sg) & (attr["MODALIDADE"] == md) & (livre_cap > 0.01)
        mover = min(sobra[idx].sum(), livre_cap[viz].sum())
        if mover <= 0:
            continue
        ident.loc[idx, "RECEITA_APE_BRL"] += mover * sobra[idx] / sobra[idx].sum()
        ident.loc[viz[viz].index, "RECEITA_APE_BRL"] -= mover * livre_cap[viz] / livre_cap[viz].sum()

    # reduction per group: the net value, capped at the exemption capacity
    reducao = (-ident["RECEITA_APE_BRL"]).clip(upper=cap_g)
    escala = (reducao / cap_g.replace(0, np.nan)).fillna(0.0)
    for c in CAMADAS:
        APE_RED[c][COMP_TUSD] = DIF_APE[c].mul(qtd_g * escala, axis=0)
    ident["AJUSTE_APE_BRL"] = -APE_RED["TA"].sum(axis=1)
    ident["RECEITA_BRL"] = ident["RECEITA_BRL"] + ident["RECEITA_APE_BRL"]

    excesso = (-ident["RECEITA_APE_BRL"] - cap_g).clip(lower=0.0)
    print(f"regime APE: {len(APE_USADAS)} linhas do BW na base | receita APE liquida "
          f"R$ {ident['RECEITA_APE_BRL'].sum():,.2f} | abatimento nos itens TUSD R$ {ident['AJUSTE_APE_BRL'].sum():,.2f}")
    if excesso.sum() > 0.01:
        print(f"   ATENCAO: abatimento travado na capacidade em {int((excesso > 0.01).sum())} grupo(s); "
              f"excesso R$ {excesso.sum():,.2f} segue para as passadas seguintes")
    if sem_destino:
        print(f"   ATENCAO: {len(sem_destino)} linha(s) APE sem grupo de destino ficam fora da conta "
              f"(R$ {ape_linhas.loc[sem_destino, 'RECEITA_BRL'].sum():,.2f})")

# --- first pass: the free-market gap is absorbed by Fio A + Fio B alone.
fio_ta = B1_GRUPO["TA"][COLS_FIO].sum(axis=1)
falta  = ident["CALCULADO_TA"] + ident["AJUSTE_APE_BRL"] - ident["RECEITA_BRL"]
desc   = np.where(em_fio & (fio_ta != 0), falta / fio_ta.replace(0, np.nan), 0.0)
desc   = np.clip(np.nan_to_num(desc), 0.0, 1.0)   # a discount lives between 0% and 100%
ident["DESC_FIO"] = desc

B2_GRUPO = {}
for camada in CAMADAS:
    m = B1_GRUPO[camada] - APE_RED[camada]
    m[COLS_FIO] = m[COLS_FIO].mul(1.0 - desc, axis=0)
    B2_GRUPO[camada] = m

# How much the first pass alone moved, in reais - only regime "fio" groups
# move here, by construction of the discount above.
parcial = B2_GRUPO["TA"].sum(axis=1)
ident["AJUSTE_1A_PASSADA_BRL"] = parcial - ident["CALCULADO_TA"] - ident["AJUSTE_APE_BRL"]

# --- second pass: whatever is left is spread proportionally across every
# component, so the block closes exactly against the BW revenue. For regime
# "uniforme" (no discount mechanism in the first pass) this IS the whole
# adjustment - expected, not a deviation. For regime "fio", a nonzero value
# here means the 0%-100% clamp on the discount kept the first pass from
# closing the group alone - that IS a signal worth looking at.
sem_ajuste = parcial == 0
fator2     = np.where(~sem_ajuste, ident["RECEITA_BRL"] / parcial.replace(0, np.nan), 0.0)
fator2     = np.nan_to_num(fator2)
if not APLICAR_2A_PASSADA:
    # second pass switched off (Passo 0): the block stops after the first pass and the
    # remaining gap is reported, not spread
    fator2 = np.ones(len(ident))
B2_GRUPO   = {c: B2_GRUPO[c].mul(fator2, axis=0) for c in CAMADAS}
ident["PCT_SEGUNDA_PASSADA"]   = np.where(sem_ajuste, np.nan, fator2 - 1.0)
ident["AJUSTE_2A_PASSADA_BRL"] = B2_GRUPO["TA"].sum(axis=1) - parcial

# gap left in the block: zero for every adjustable group when the second pass runs
ident["DIF_NAO_AJUSTADA_BRL"] = ident["RECEITA_BRL"] - B2_GRUPO["TA"].sum(axis=1)

# line-level view of the gap (only needed when the second pass is off): each BW line
# keeps its own bloco 1 and receives the first-pass discount of its group
DIF_LINHAS = None
if not APLICAR_2A_PASSADA:
    desc_linha = codigo_grupo.map(pd.Series(desc, index=ident.index))
    b1_linha = BLOCO1["TA"][COMPONENTES]
    b2_linha = b1_linha.copy()
    b2_linha[COLS_FIO] = b2_linha[COLS_FIO].mul(1.0 - desc_linha, axis=0)
    DIF_LINHAS = tarifaveis[["LINHA_EXCEL", "CATEGORIA_TARIFA", "FLAG_MERCADO", "CLASSE_PRINCIPAL",
                             "CLASSE_SUBCLASSE", "GRP_MONT", "GRP_QUANT", "GRP_QUANT_TXT",
                             "OPERACAO", "OPERACAO_TXT", "SUBOPERACAO", "SUBOPERACAO_TXT", "CHAVE", "TIPO_FATURAMENTO",
                             "QTD_CONSUMO_TUSD", "QTD_CONSUMO_TE", "DEMANDA_KW", "RECEITA_BRL"]].copy()
    DIF_LINHAS.insert(0, "GRUPO", codigo_grupo.values + 1)
    DIF_LINHAS["REGIME"] = codigo_grupo.map(ident["REGIME"]).values
    DIF_LINHAS["BLOCO1_TA"] = b1_linha.sum(axis=1)
    DIF_LINHAS["DESC_FIO"] = desc_linha.values
    DIF_LINHAS["AJUSTE_1A_PASSADA_BRL"] = b2_linha.sum(axis=1) - DIF_LINHAS["BLOCO1_TA"]
    DIF_LINHAS["BLOCO2_TA"] = b2_linha.sum(axis=1)
    DIF_LINHAS["DIFERENCA_BRL"] = DIF_LINHAS["RECEITA_BRL"] - DIF_LINHAS["BLOCO2_TA"]
    if len(APE_USADAS):
        # APE lines: revenue as booked; block 2 = their share of the APE reduction of each group
        fator_ape = (ident["AJUSTE_APE_BRL"] / ident["RECEITA_APE_BRL"].replace(0, np.nan)).fillna(0.0)
        extra = bw.loc[APE_USADAS, [c for c in DIF_LINHAS.columns if c in bw.columns]].copy()
        extra.insert(0, "GRUPO", [max(APE_LINHA_GRUPO[i], key=lambda t: abs(t[1]))[0] + 1 for i in APE_USADAS])
        extra["REGIME"] = "APE"
        extra["BLOCO1_TA"] = 0.0
        extra["DESC_FIO"] = 0.0
        extra["AJUSTE_1A_PASSADA_BRL"] = 0.0
        extra["BLOCO2_TA"] = [sum(v * fator_ape[g] for g, v in APE_LINHA_GRUPO[i]) for i in APE_USADAS]
        extra["DIFERENCA_BRL"] = extra["RECEITA_BRL"] - extra["BLOCO2_TA"]
        DIF_LINHAS = pd.concat([DIF_LINHAS, extra[DIF_LINHAS.columns]], ignore_index=True)
    d = abs(DIF_LINHAS["DIFERENCA_BRL"].sum() - ident["DIF_NAO_AJUSTADA_BRL"].sum())
    assert d < 0.01, f"diferenca linha a linha nao fecha com a do grupo ({d})"
    print(f"2a passada DESLIGADA: diferenca nao ajustada R$ {ident['DIF_NAO_AJUSTADA_BRL'].sum():,.2f} "
          f"em {len(DIF_LINHAS)} linhas do BW ({len(ident)} grupos)")

# total adjustment applied to the group, whatever the route taken.
# DIFERENCA_BRL is the gap in reais: negative means the BW booked less than
# the homologated tariff would charge.
ident["DIFERENCA_BRL"] = ident["RECEITA_BRL"] - ident["CALCULADO_TA"]
ident["FATOR_AJUSTE"] = np.where(ident["CALCULADO_TA"] != 0,
                                 ident["RECEITA_BRL"] / ident["CALCULADO_TA"], np.nan)
ident["PCT_AJUSTE"] = ident["FATOR_AJUSTE"] - 1.0

# sanity: the two passes together must reproduce the total gap exactly, for
# every group that COULD be adjusted. sem_ajuste (bloco 1 zerado, see below)
# is the known exception: there is no tariff item to scale there, so their
# gap is neither closed nor moved by either pass - it stays outside the
# block, already reported below as "residuo nao ajustavel".
d = (ident["AJUSTE_APE_BRL"] + ident["AJUSTE_1A_PASSADA_BRL"] + ident["AJUSTE_2A_PASSADA_BRL"]
     - ident["DIFERENCA_BRL"])
d_ajustaveis = d[~sem_ajuste].abs().max()
if APLICAR_2A_PASSADA:
    print(f"1a passada + 2a passada = diferenca total, grupos ajustaveis, diferenca maxima: {d_ajustaveis:.10f}")

n_fio = int(em_fio.sum())
print(f"grupos: {len(ident)} | regime fio: {n_fio} | regime uniforme: {len(ident) - n_fio}")
print(f"sem ajuste possivel (bloco 1 zerado): {int(sem_ajuste.sum())}")
print(f"ajuste medio ponderado: {100*(ident['RECEITA_BRL'].sum()/ident['CALCULADO_TA'].sum()-1):+.2f}%")

emfio = ident[em_fio]
print(f"\ndesconto sobre Fio A + Fio B, regime fio: "
      f"minimo {100*emfio['DESC_FIO'].min():.1f}% | mediana {100*emfio['DESC_FIO'].median():.1f}% "
      f"| maximo {100*emfio['DESC_FIO'].max():.1f}%")

# --- resumo da 2a passada: separa desconto de verdade (1a passada, so existe
# no regime fio) de fechamento de conta (2a passada). No regime uniforme a 2a
# passada e o proprio mecanismo de ajuste, sem 1a passada - nao e desvio. No
# regime fio, uma 2a passada != 0 significa que o clamp 0%-100% nao deixou a
# 1a passada fechar o grupo sozinha - listado linha a linha abaixo.
total_1a = ident["AJUSTE_1A_PASSADA_BRL"].sum()
total_2a = ident["AJUSTE_2A_PASSADA_BRL"].sum()
print(f"\nresumo do bloco 2:")
print(f"  regime APE (isencao de encargos, itens TUSD) ............. R$ {ident['AJUSTE_APE_BRL'].sum():,.2f}")
print(f"  1a passada (desconto de fonte incentivada, regime fio) .... R$ {total_1a:,.2f}")
print(f"  2a passada (fechamento de conta, todas as linhas) ......... R$ {total_2a:,.2f}")

if not APLICAR_2A_PASSADA:
    # second pass off: report the gap left by regime instead of the second-pass diagnostics
    print("\n2a passada DESLIGADA - diferenca nao ajustada por regime:")
    for reg, v in ident.groupby("REGIME")["DIF_NAO_AJUSTADA_BRL"].sum().items():
        print(f"   regime {reg:<9} R$ {v:,.2f}")
fio_2a = ident.loc[em_fio, "AJUSTE_2A_PASSADA_BRL"]
residuo_fio = fio_2a[fio_2a.abs() > 0.01]
if not APLICAR_2A_PASSADA:
    pass
elif len(residuo_fio):
    print(f"\nATENCAO: {len(residuo_fio)} grupo(s) do regime fio precisaram de 2a passada "
          f"(o desconto sozinho nao fechou), total R$ {residuo_fio.sum():,.2f}:")
    for idx in residuo_fio.abs().sort_values(ascending=False).index:
        r = ident.loc[idx]
        pct = 100 * r["AJUSTE_2A_PASSADA_BRL"] / r["RECEITA_BRL"] if r["RECEITA_BRL"] else 0.0
        print(f"   {r['GRP_QUANT']} | {r['CATEGORIA_TARIFA']} | {r['CLASSE_SUBCLASSE']} | "
              f"R$ {r['AJUSTE_2A_PASSADA_BRL']:,.2f} ({pct:+.2f}% da receita do grupo)")
else:
    print("\nnenhum grupo do regime fio precisou de 2a passada - o desconto sozinho fechou todos")

uniforme_2a = ident.loc[~em_fio, "AJUSTE_2A_PASSADA_BRL"]
if APLICAR_2A_PASSADA:
    print(f"\n2a passada no regime uniforme (mecanismo normal de ajuste, nao e desvio): "
          f"{(uniforme_2a.abs() > 0.01).sum()} grupo(s) movidos, total R$ {uniforme_2a.sum():,.2f}")

extremos = ident.loc[~sem_ajuste, ["CATEGORIA_TARIFA", "FLAG_MERCADO", "GRP_QUANT", "REGIME",
                                   "RECEITA_BRL", "CALCULADO_TA", "DESC_FIO", "PCT_AJUSTE"]]
print("\nmaiores ajustes, em valor absoluto:")
extremos = extremos.assign(DIF=(extremos["RECEITA_BRL"] - extremos["CALCULADO_TA"]).abs())
extremos.nlargest(5, "DIF").drop(columns="DIF")


---
## Passo 10 — Conferências

Todas têm de passar antes de a planilha ser usada; o resultado aparece no fim da célula.

| Conferência | O que garante |
|---|---|
| 1. Bloco 2 = receita, grupo a grupo | A abertura fecha no faturado (com a 2ª passada desligada: bloco 2 + diferença = receita) |
| 1b. Resíduo não ajustável | Mostra, em reais, a receita de grupos com cálculo zero, que não tem item para escalar e fica fora do bloco 2 |
| 1c. Abatimento APE ≤ desconto registrado | A isenção não cria desconto inexistente |
| 2. BE + CVA + FIN = TA | A identidade tarifária sobreviveu aos ajustes |
| 3. Famílias = soma dos itens | Nenhum item ficou sem família |
| 4. Tarifável + não tarifável = total do BW | Nada se perdeu na separação |

O item 5 não é aprovação: mostra o efeito do regime `fio` sobre encargos e perdas dos livres do
grupo A, comparado com o que um rateio uniforme faria.

In [ ]:
ok = True

# 1. o bloco 2 soma exatamente a receita registrada
soma_b2 = B2_GRUPO["TA"].sum(axis=1)
if APLICAR_2A_PASSADA:
    d = (soma_b2 - ident["RECEITA_BRL"]).where(~sem_ajuste).abs().max()
    print(f"1. bloco 2 (TA) x receita, grupos ajustaveis .. diferenca maxima {d:,.6f}")
else:
    # without the second pass the block does not close by design: the check is that
    # block + reported gap reproduce the revenue
    d = (soma_b2 + ident["DIF_NAO_AJUSTADA_BRL"] - ident["RECEITA_BRL"]).abs().max()
    print(f"1. bloco 2 (TA) + diferenca nao ajustada x receita .. diferenca maxima {d:,.6f}"
          f"  (2a passada desligada: diferenca R$ {ident['DIF_NAO_AJUSTADA_BRL'].sum():,.2f})")
ok &= d < 0.01

# 1b. o residuo que o ajuste NAO consegue fechar: grupos com calculo zero.
#     Nao ha tarifa para escalar, entao a receita desses grupos fica de fora
#     do bloco 2. E preciso enxergar quanto e, nao esconder.
residuo = ident["RECEITA_BRL"].where(sem_ajuste, 0.0).sum()
print(f"1b. residuo nao ajustavel ({int(sem_ajuste.sum())} grupos) ......... R$ {residuo:,.2f}")

# 1c. APE regime: reduction only on TUSD items, never above the net APE revenue in absolute value
if REGIME_APE:
    # |reduction| <= |net APE revenue| in every group with a net discount
    d = (ident["RECEITA_APE_BRL"] - ident["AJUSTE_APE_BRL"]).where(ident["RECEITA_APE_BRL"] < 0, -1.0).max()
    print(f"1c. regime APE: abatimento nao passa do desconto ........ excesso maximo {max(d, 0.0):,.6f}")
    ok &= d < 0.01

# 2. BE + CVA + FIN = TA, no ajustado
d = (B2_GRUPO["TA"] - B2_GRUPO["BE"] - B2_GRUPO["CVA"] - B2_GRUPO["FIN"]).abs().max().max()
print(f"2. BE + CVA + FIN = TA no bloco 2 ............. diferenca maxima {d:,.10f}")
ok &= d < 0.01

# 3. a soma das familias reproduz a soma dos componentes
d = max(abs(B1_GRUPO[c][COMPONENTES].sum().sum() -
            sum(B1_GRUPO[c][lista].sum().sum() for lista in FAMILIA.values()))
        for c in CAMADAS)
print(f"3. familias x componentes, bloco 1 ............ diferenca maxima {d:,.6f}")
ok &= d < 0.01

# 4. tarifavel + fora da conta = receita total do arquivo
# APE lines used by the APE regime are inside the groups (Passo 9), not outside the account
fora = bw[~bw.index.isin(tarifaveis.index) & ~bw.index.isin(APE_USADAS)].copy()
fora["SITUACAO"] = np.where(fora["TARIFAVEL"], "SEM TARIFA", "NÃO")
total = ident["RECEITA_BRL"].sum() + fora["RECEITA_BRL"].sum()
d = abs(total - bw["RECEITA_BRL"].sum())
print(f"4. tarifavel + nao tarifavel x total do BW .... diferenca {d:,.6f}")
ok &= d < 0.01

# 5. medida, nao aprovacao: quanto dos encargos foi preservado no regime fio.
#    Sob rateio uniforme eles seriam cortados junto com o fio, o que e errado.
fio_reg = ident["REGIME"] == "fio"
enc = FAMILIA["ENCARGOS"] + FAMILIA["PERDAS"]
enc = [c for c in enc if c in B1_GRUPO["TA"].columns]
e1 = B1_GRUPO["TA"].loc[fio_reg, enc].sum().sum()
e2 = B2_GRUPO["TA"].loc[fio_reg, enc].sum().sum()
unif = (B1_GRUPO["TA"].loc[fio_reg, enc]
        .mul(ident.loc[fio_reg, "FATOR_AJUSTE"].fillna(0.0), axis=0).sum().sum())
print(f"\n5. encargos + perdas dos clientes livres do grupo A")
print(f"   sem ajuste ............ R$ {e1:,.2f}")
print(f"   após ajustes do regime fio (livres AT) ...... R$ {e2:,.2f}")
print(f"   caso fosse usado o rateio uniforme (ajustar todos os itens tarifarios)... R$ {unif:,.2f}  (impacto total do ajuste fio R$ {e2 - unif:,.2f})")

print("\nTODAS AS CONFERENCIAS PASSARAM" if ok else "\nALGUMA CONFERENCIA FALHOU - NAO USAR")

---
# Parte II — Geradoras, suprimentos e uso da distribuição (SD)

Esses clientes não passam pelo BW: são faturados numa planilha de controle própria, uma aba por
cliente, e entram na mesma tabela de saída, identificados na coluna `Source`.

O SD tem uma vantagem sobre o fornecimento: **a planilha informa a tarifa efetivamente aplicada em
cada mês**, então não há vigência a estimar. O que falta é abrir essa tarifa nos 37 itens: a receita
de cada linha é distribuída na proporção de cada item dentro da tarifa homologada da mesma chave
(Passo 4), já ponderada por vigência. O total fica exato e o SD sai aberto nas quatro camadas.

A aba `SD tarifas` da planilha é usada para resolver a chave do CPFL, que não vem na aba do cliente, e
como recurso quando a tabela homologada tem o lado da linha zerado (caso do lado TE da Ceris): nesse
caso só há tarifa de aplicação, sem as camadas BE, CVA e FIN.

---
## Passo 11 — Classificação dos clientes

A aba `Permissionária e CUSD` traz a classificação de cada cliente, mas chega oculta e sem garantia
de que venha em toda entrega. Por isso a classificação fica aqui no código, e a aba serve só para
conferir. Cliente que aparecer na planilha e não estiver nesta lista é reportado, nunca adivinhado.

In [ ]:
# Client -> type. This is the authority; the workbook is only cross-checked against it.
DE_PARA_CLIENTE = {
    "PCH Rasgão (EMAE)":  "GERAÇÃO",
    "EE Pedreira (EMAE)": "GERAÇÃO",
    "EE Traição (EMAE)":  "GERAÇÃO",
    "Petrobrás":          "GERAÇÃO",
    "PCH Pirapora":       "GERAÇÃO",
    "UTE São João":       "GERAÇÃO",
    "Tecipar":            "GERAÇÃO",
    "Ceris":              "SUPRIMENTO",
    "Neoenergia Elektro": "USO DISTRIBUIÇÃO",
    "CPFL Piratininga":   "USO DISTRIBUIÇÃO",
}

# The consolidated sheet names the same clients differently.
ALIAS_CONSOLIDADO = {"CERIS": "Ceris", "CPFL": "CPFL Piratininga",
                     "ELEKTRO": "Neoenergia Elektro"}

# Invoice issue is one month after the consumption month: these clients bill on the
# last day of the month, so the invoice never falls inside its own competence. The consolidated
# sheet and the management report use the invoice month too (e.g. EE Pedreira: consumption of
# Jan/26, billed in Feb/26, is the Feb/26 revenue). Sheets without an issue date (CERIS, Usinas
# Geradoras) use their own month column.
DEFASAGEM_EMISSAO = 1

print(f"{len(DE_PARA_CLIENTE)} clientes no de-para")
for tipo in ("GERAÇÃO", "SUPRIMENTO", "USO DISTRIBUIÇÃO"):
    quais = [c for c, t in DE_PARA_CLIENTE.items() if t == tipo]
    print(f"  {tipo:<18}{len(quais)}: {', '.join(quais)}")

---
## Passo 12 — Ler a planilha do SD

Cada aba tem um desenho próprio: colunas em posições diferentes, chave tarifária numa linha acima do
cabeçalho, tarifa impressa em umas e implícita em outras. O layout de cada aba está declarado abaixo,
num único lugar.

**Mês de referência.** É o mês de emissão da fatura, não o de consumo. Quando a aba não traz a
emissão, usa-se o mês de consumo mais um; a Ceris é a exceção e usa o mês de competência da própria
aba.

**PIS, COFINS e ultrapassagem ficam de fora.** A receita usada é a líquida de tributos, e a
ultrapassagem não tem item na tarifa. As três rubricas são lidas e reportadas, para que a conferência
contra a aba consolidada as desconte de forma explícita.

In [ ]:
import datetime as dt

def _item(qty, key, gr, posto, un, fator, tar, tar_fator, rec, comp, rub, uc=None,
          no_calculo=True):
    # no_calculo=False: the line is read and reported, but stays out of the
    # tariff opening. Excess demand is billed by a regulatory rule, not by a
    # component of the tariff table, and the area receives it from another source.
    return dict(qty_col=qty, key_col=key, key_uc=uc, grandeza=gr, posto=posto,
                unidade=un, qty_factor=fator, tarifa_col=tar, tarifa_factor=tar_fator,
                receita_col=rec, componente=comp, rubrica=rub, no_calculo=no_calculo)

_EMAE = [
    _item(10, 10, "DEMANDA", "DP",  "kW",  1.0,   12, 1.0,    15, "TUSD", "Demanda faturada ponta"),
    _item(11, 11, "DEMANDA", "DFP", "kW",  1.0,   13, 1.0,    16, "TUSD", "Demanda faturada fora ponta"),
    _item( 4,  4, "CONSUMO", "EP",  "MWh", 0.001, 14, 1000.0, 17, "TUSD", "Consumo ponta"),
    _item( 5,  5, "CONSUMO", "EFP", "MWh", 0.001, 14, 1000.0, 18, "TUSD", "Consumo fora ponta"),
]
_USO = [
    _item(10, 10, "DEMANDA", "DP",  "kW",  1.0,   12, 1.0,    15, "TUSD", "Demanda faturada ponta"),
    _item(11, 11, "DEMANDA", "DFP", "kW",  1.0,   13, 1.0,    16, "TUSD", "Demanda faturada fora ponta"),
    _item( 4,  4, "CONSUMO", "E",   "MWh", 0.001, 14, 1000.0, 17, "TUSD", "Consumo ponta"),
    _item( 5,  5, "CONSUMO", "E",   "MWh", 0.001, 14, 1000.0, 18, "TUSD", "Consumo fora ponta"),
]
_CPFL = [dict(d, key_col=None, key_uc="CPFL Piratininga") for d in _USO]

ABAS_SD = [
    dict(sheet="Rasgão (EMAE) -203620649 ", cliente="PCH Rasgão (EMAE)",
         key_row=3, header_row=4, first_data_row=5, date_col=1, date_basis="EMISSAO",
         emissao_col=9, pis_col=11, cofins_col=12,
         items=[_item(6, 6, "DEMANDA", "D", "kW", 1.0, 7, 1.0, 8, "TUSD", "Demanda faturada")]),
    dict(sheet="Pedreira (EMAE) - 203629904", cliente="EE Pedreira (EMAE)",
         key_row=1, header_row=2, first_data_row=3, date_col=1, date_basis="EMISSAO",
         emissao_col=20, pis_col=22, cofins_col=23, items=_EMAE),
    dict(sheet="Traição (EMAE) Novo - 203948143", cliente="EE Traição (EMAE)",
         key_row=1, header_row=2, first_data_row=3, date_col=1, date_basis="EMISSAO",
         emissao_col=20, pis_col=22, cofins_col=23, items=_EMAE),
    dict(sheet="ELEKTRO - 203636054", cliente="Neoenergia Elektro",
         key_row=2, header_row=3, first_data_row=4, date_col=1, date_basis="EMISSAO",
         emissao_col=20, pis_col=22, cofins_col=23, items=_USO),
    dict(sheet="CPFL - 203631034", cliente="CPFL Piratininga",
         key_row=None, header_row=1, first_data_row=2, date_col=1, date_basis="EMISSAO",
         emissao_col=20, pis_col=22, cofins_col=23, items=_CPFL),
    dict(sheet="CERIS", cliente="Ceris",
         key_row=2, header_row=3, first_data_row=4, date_col=2, date_basis="MES",
         emissao_col=None, pis_col=17, cofins_col=18,
         items=[_item( 4, 4, "DEMANDA", "DP",  "kW",  1.0,   None, 1.0, 11, "TUSD", "Demanda ponta"),
                _item( 5, 5, "DEMANDA", "DFP", "kW",  1.0,   None, 1.0, 12, "TUSD", "Demanda fora ponta"),
                _item( 6, 4, "DEMANDA", "DP",  "kW",  1.0,   None, 1.0, 13, "TUSD", "Ultrapassagem ponta",      no_calculo=False),
                _item( 7, 5, "DEMANDA", "DFP", "kW",  1.0,   None, 1.0, 14, "TUSD", "Ultrapassagem fora ponta", no_calculo=False),
                _item(10, 8, "CONSUMO", "E",   "MWh", 0.001, None, 1.0, 15, "TE",   "Energia sem Proinfa"),
                _item(10, 8, "CONSUMO", "E",   "MWh", 0.001, None, 1.0, 16, "TUSD", "Energia sem Proinfa")]),
]

# Quantity-only sheet: these four generators have no sheet of their own.
ABA_MATRIZ = dict(sheet="Usinas Geradoras", key_row=4, header_row=5, first_data_row=6,
                  info_col=2, date_col=3, primeira_col=4, ultima_col=7,
                  info_map={"Demanda kW": ("DEMANDA", "D", "kW", 1.0, "TUSD", "Demanda contratada")})
ABA_CONSOLIDADO = dict(sheet="Permissionária e CUSD", header_row=4, first_data_row=5,
                       info_col=2, date_col=3, primeira_col=4, ultima_col=13)
print(f"{len(ABAS_SD)} abas de cliente declaradas")

In [ ]:
def _data(v):
    """Accept a real date or the dd.mm.yyyy text some sheets use."""
    if isinstance(v, dt.datetime): return v.date()
    if isinstance(v, dt.date):     return v
    if isinstance(v, str):
        m = re.match(r"^\s*(\d{2})[./-](\d{2})[./-](\d{4})", v)
        if m: return dt.date(int(m.group(3)), int(m.group(2)), int(m.group(1)))
    return None

def _mes(d):      return dt.date(d.year, d.month, 1) if d else None
def _soma_mes(d, n):
    if not d: return None
    t = d.year * 12 + d.month - 1 + n
    return dt.date(t // 12, t % 12 + 1, 1)
def _num(v):
    try:
        f = float(v)
        return f if np.isfinite(f) else 0.0
    except (TypeError, ValueError):
        return 0.0

def le_tarifas_sd(livro):
    """Components by key, from the "SD tarifas" sheet.
    Layout: 1 blank | 2-9 attributes | 10 key | 11 Total TUSD | 12 Total TE | 13+ components."""
    linhas = list(livro["SD tarifas"].iter_rows(values_only=True))
    cab = [str(c).strip() if c is not None else "" for c in linhas[0]]
    nomes = [c for c in cab[12:] if c]
    reg = {}
    for l in linhas[1:]:
        if not l[9]: continue
        reg[str(l[9]).strip()] = dict(
            uc=l[6], posto=l[7], total_tusd=_num(l[10]), total_te=_num(l[11]),
            comp={n: _num(l[12 + i]) for i, n in enumerate(nomes)})
    return nomes, reg

def le_sd(caminho):
    livro = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    nomes, tarifas = le_tarifas_sd(livro)
    por_uc = {(r["uc"], r["posto"]): k for k, r in tarifas.items()}
    reg = []

    def guarda(**kw):
        reg.append(kw)

    for aba in ABAS_SD:
        linhas = list(livro[aba["sheet"]].iter_rows(values_only=True))
        krow = linhas[aba["key_row"] - 1] if aba["key_row"] else None
        for l in linhas[aba["first_data_row"] - 1:]:
            mes = _mes(_data(l[aba["date_col"] - 1]))
            if not mes: continue
            if aba["date_basis"] == "EMISSAO":
                ref = _mes(_data(l[aba["emissao_col"] - 1])) if aba["emissao_col"] else None
                origem = "data de emissão"
                if not ref:
                    ref, origem = _soma_mes(mes, DEFASAGEM_EMISSAO), "competência + defasagem"
            else:
                ref, origem = mes, "mês de competência"
            for it in aba["items"]:
                qtd = _num(l[it["qty_col"] - 1]) * it["qty_factor"]
                rec = _num(l[it["receita_col"] - 1])
                if qtd == 0 and rec == 0: continue
                chave = (str(krow[it["key_col"] - 1] or "").strip() if it["key_col"] and krow
                         else por_uc.get((it["key_uc"], it["posto"]), ""))
                if it["tarifa_col"]:
                    tar, fonte = _num(l[it["tarifa_col"] - 1]) * it["tarifa_factor"], "planilha"
                else:
                    tar, fonte = (rec / qtd if qtd else 0.0), "implícita"
                guarda(REFERENCIA=ref, MES_CONSUMO=mes, ORIGEM_DATA=origem,
                       CLIENTE=aba["cliente"], ABA=aba["sheet"], CHAVE=chave,
                       GRANDEZA=it["grandeza"], POSTO=it["posto"], UNIDADE=it["unidade"],
                       RUBRICA=it["rubrica"], COMPONENTE=it["componente"], QUANTIDADE=qtd,
                       TARIFA_APLICADA=tar, FONTE_TARIFA=fonte, RECEITA_BRL=rec,
                       FONTE_RECEITA="aba do cliente", NO_CALCULO=it["no_calculo"])

    m = ABA_MATRIZ
    linhas = list(livro[m["sheet"]].iter_rows(values_only=True))
    cabec, krow = linhas[m["header_row"] - 1], linhas[m["key_row"] - 1]
    for l in linhas[m["first_data_row"] - 1:]:
        info = str(l[m["info_col"] - 1] or "").strip()
        ref = _mes(_data(l[m["date_col"] - 1]))
        if not ref or info not in m["info_map"]: continue
        gr, posto, un, fator, comp, rub = m["info_map"][info]
        for c in range(m["primeira_col"], m["ultima_col"] + 1):
            cliente, qtd = str(cabec[c - 1] or "").strip(), _num(l[c - 1]) * fator
            if not cliente or qtd == 0: continue
            guarda(REFERENCIA=ref, MES_CONSUMO=ref, ORIGEM_DATA="mês de competência",
                   CLIENTE=cliente, ABA=m["sheet"], CHAVE=str(krow[c - 1] or "").strip(),
                   GRANDEZA=gr, POSTO=posto, UNIDADE=un, RUBRICA=rub, COMPONENTE=comp,
                   QUANTIDADE=qtd, TARIFA_APLICADA=np.nan, FONTE_TARIFA="ausente",
                   RECEITA_BRL=np.nan, FONTE_RECEITA="pendente", NO_CALCULO=True)

    c = ABA_CONSOLIDADO
    linhas = list(livro[c["sheet"]].iter_rows(values_only=True))
    cabec = linhas[c["header_row"] - 1]
    cons = []
    for l in linhas[c["first_data_row"] - 1:]:
        info = str(l[c["info_col"] - 1] or "").strip()
        ref = _mes(_data(l[c["date_col"] - 1]))
        if not ref or not info: continue
        for col in range(c["primeira_col"], c["ultima_col"] + 1):
            nome = str(cabec[col - 1] or "").strip()
            if nome and l[col - 1] is not None:
                cons.append(dict(REFERENCIA=ref, INFORMACAO=info,
                                 CLIENTE=ALIAS_CONSOLIDADO.get(nome, nome),
                                 VALOR=_num(l[col - 1])))
    livro.close()
    return pd.DataFrame(reg), pd.DataFrame(cons), nomes, tarifas

sd_bruto, sd_consolidado, COMP_SD, TARIFA_SD = le_sd(ARQ_SD)
sd_bruto["TIPO"] = sd_bruto["CLIENTE"].map(DE_PARA_CLIENTE)

print(f"{len(sd_bruto)} linhas lidas | {sd_bruto['CLIENTE'].nunique()} clientes "
      f"| {len(TARIFA_SD)} chaves na aba SD tarifas")
orfaos = sorted(sd_bruto.loc[sd_bruto["TIPO"].isna(), "CLIENTE"].unique())
print("clientes fora do de-para:", ", ".join(orfaos) if orfaos else "nenhum")
sem_tarifa = sorted(set(sd_bruto.loc[sd_bruto["CHAVE"] != "", "CHAVE"]) - set(TARIFA_SD))
print("chaves sem tarifa na aba SD tarifas:", len(sem_tarifa))

---
## Passo 13 — Abrir a receita do SD

- A receita dos geradores sem aba própria vem da aba consolidada, líquida de tributos.
- Quando a aba não imprime a tarifa, ela é obtida pela receita dividida pela quantidade.
- A receita é aberta na proporção de cada item dentro da tarifa homologada da mesma chave; quando a
  tabela homologada tem aquele lado zerado, a proporção vem da aba `SD tarifas`, só na tarifa de
  aplicação.
- Linhas de PIS e COFINS não são abertas: entram como não tarifáveis, como os tributos do BW.

In [ ]:
alvo = dt.date(int(REFERENCIA[:4]), int(REFERENCIA[5:7]), 1)
sd = sd_bruto[sd_bruto["REFERENCIA"] == alvo].copy()

# revenue of the four matrix generators comes from the consolidated sheet
# the consolidated sheet reports revenue gross of PIS/COFINS; net it before use
_no_mes = sd_consolidado[sd_consolidado["REFERENCIA"] == alvo]
_por_info = (_no_mes.pivot_table(index="CLIENTE", columns="INFORMACAO",
                                 values="VALOR", aggfunc="sum").fillna(0.0))
for _c in ("Receita", "PIS", "COFINS"):
    if _c not in _por_info: _por_info[_c] = 0.0
receita_cons = _por_info["Receita"] - _por_info["PIS"] - _por_info["COFINS"]
falta = sd["RECEITA_BRL"].isna()
sd.loc[falta, "RECEITA_BRL"] = sd.loc[falta, "CLIENTE"].map(receita_cons)
sd.loc[falta, "FONTE_RECEITA"] = "aba consolidada, líquida de tributos"
sem_tar = sd["TARIFA_APLICADA"].isna() & (sd["QUANTIDADE"] != 0)
sd.loc[sem_tar, "TARIFA_APLICADA"] = sd.loc[sem_tar, "RECEITA_BRL"] / sd.loc[sem_tar, "QUANTIDADE"]
sd.loc[sem_tar, "FONTE_TARIFA"] = "implícita"
sd["RECEITA_BRL"] = sd["RECEITA_BRL"].fillna(0.0)

# lines kept out of the tariff opening, reported as a reconciling item
sd_fora = sd[~sd["NO_CALCULO"]].copy()
sd = sd[sd["NO_CALCULO"]].reset_index(drop=True)
if len(sd_fora):
    print("fora do calculo, por regra:")
    for _, r in sd_fora.iterrows():
        print(f"   {r['CLIENTE']} | {r['RUBRICA']} | R$ {r['RECEITA_BRL']:,.2f}")
    print(f"   total R$ {sd_fora['RECEITA_BRL'].sum():,.2f}")


# --- breakdown: the revenue split by the shares of the homologated tariff table ---------
# TUSD components price consumption/demand of the network, TE components the energy.
# Scaling every layer by the same factor keeps TA = BE + CVA + FIN.
COLS_TUSD = [c for c in COMPONENTES if not c.startswith("TE")]
COLS_TE   = [c for c in COMPONENTES if c.startswith("TE")]

B1_SD_CAMADAS = {c: pd.DataFrame(0.0, index=sd.index, columns=COMPONENTES) for c in CAMADAS}
sem_chave, recorreu = [], []
for i, r in sd.iterrows():
    chave, lado = r["CHAVE"], r["COMPONENTE"]
    if chave not in TARIFA["TA"].index:
        sem_chave.append((chave, r["CLIENTE"], r["RUBRICA"]))
        continue
    cols = COLS_TUSD if lado == "TUSD" else COLS_TE
    total = TARIFA["TA"].loc[chave, cols].sum()
    if total:
        fator = r["RECEITA_BRL"] / total
        for camada in CAMADAS:
            B1_SD_CAMADAS[camada].loc[i, cols] = TARIFA[camada].loc[chave, cols].values * fator
        continue
    # Tariff table zeroed on this side: fall back to the "SD tarifas" sheet, which
    # carries only the tariff of application. The other layers stay empty.
    ref = TARIFA_SD.get(chave)
    # sum the components, not the published total, which is rounded to two decimals
    total_sd = sum(v for n, v in ref["comp"].items() if n in cols) if ref else 0.0
    if not total_sd:
        sem_chave.append((chave, r["CLIENTE"], r["RUBRICA"]))
        continue
    fator = r["RECEITA_BRL"] / total_sd
    for n, v in ref["comp"].items():
        if n in cols:
            B1_SD_CAMADAS["TA"].loc[i, n] = v * fator
    recorreu.append((r["CLIENTE"], r["RUBRICA"], lado, r["RECEITA_BRL"]))

B1_SD = B1_SD_CAMADAS["TA"]
if sem_chave:
    print("ATENCAO: linhas sem tarifa em nenhuma das duas tabelas:")
    for ch, cli, rub in sem_chave:
        print(f"   {cli} | {rub} | {ch[:60]}")
if recorreu:
    print("ATENCAO: a tabela de tarifas esta zerada neste lado; usada a aba SD tarifas,")
    print("         que so tem a tarifa de aplicacao (BE, CVA e FIN ficam vazios):")
    for cli, rub, lado, val in recorreu:
        print(f"   {cli} | {rub} | {lado} | R$ {val:,.2f}")

tarifavel_sd = pd.Series(True, index=sd.index)
print(f"referência {REFERENCIA} | {len(sd)} linhas | {sd['CLIENTE'].nunique()} clientes")
print(f"receita SD, sem tributos: R$ {sd['RECEITA_BRL'].sum():,.2f}")
print()
print(sd.groupby(["TIPO", "CLIENTE"]).agg(
    linhas=("RECEITA_BRL", "size"), receita=("RECEITA_BRL", "sum"),
    origem_receita=("FONTE_RECEITA", "first"), base_data=("ORIGEM_DATA", "first")).to_string())

---
## Passo 14 — Juntar BW e SD

O SD não precisa de ajuste: a tarifa aplicada já está na planilha, então o calculado é a própria
receita. As linhas entram com regime `direto` e diferença zero. As colunas de identificação recebem
o cliente em `Classe/Sub-classe` e o tipo em `Flag Cli.Liv./ Cat.`, conforme a classificação do
Passo 11.

In [ ]:
def _categoria_sd(chave):
    """Subgroup + modality, read off the front of the key."""
    m = re.match(r"^(A2|A3a|A4|AS|B1|B2|B3|B4a|B4b)(Azul|Verde|Convencional|Distribuição|Geração)?", str(chave))
    return f"{m.group(1)} {m.group(2) or ''}".strip() if m else ""

ident_sd = pd.DataFrame({
    "CHAVE":            sd["CHAVE"],
    "CLASSE_PRINCIPAL": "Geradoras, suprimentos e uso da distribuição",
    "CATEGORIA_TARIFA": sd["CHAVE"].map(_categoria_sd),
    "GRP_MONT":         sd["RUBRICA"],
    "GRP_QUANT":        sd["POSTO"],
    "GRP_QUANT_TXT":    sd["RUBRICA"],
    "OPERACAO":         sd["COMPONENTE"],
    "OPERACAO_TXT":     "Faturamento SD",
    "SUBOPERACAO":      sd["FONTE_TARIFA"],
    "SUBOPERACAO_TXT":  sd["ORIGEM_DATA"],
    "FLAG_MERCADO":     sd["TIPO"],
    "CLASSE_SUBCLASSE": sd["CLIENTE"],
    "TIPO_FATURAMENTO": "FAT",
    "LINHAS_ORIGEM":    sd["ABA"] + " · consumo " + sd["MES_CONSUMO"].astype(str).str[:7],
    "CONSUMO_TUSD":     np.where((sd["GRANDEZA"] == "CONSUMO") & (sd["COMPONENTE"] == "TUSD"),
                                 sd["QUANTIDADE"] * 1000.0, 0.0),
    "CONSUMO_TE":       np.where((sd["GRANDEZA"] == "CONSUMO") & (sd["COMPONENTE"] == "TE"),
                                 sd["QUANTIDADE"] * 1000.0, 0.0),
    "DEMANDA":          np.where(sd["GRANDEZA"] == "DEMANDA", sd["QUANTIDADE"], 0.0),
    "RECEITA_BRL":      sd["RECEITA_BRL"],
})
ident_sd["TOTAL_CONSUMO"] = ident_sd["CONSUMO_TUSD"] + ident_sd["CONSUMO_TE"]
ident_sd["TOTAL_DEMANDA"] = ident_sd["DEMANDA"]
ident_sd["CALCULADO_TA"]  = B1_SD.sum(axis=1)
ident_sd["DIFERENCA_BRL"] = ident_sd["RECEITA_BRL"] - ident_sd["CALCULADO_TA"]
ident_sd["FATOR_AJUSTE"]  = 1.0
ident_sd["PCT_AJUSTE"]    = 0.0
ident_sd["REGIME"]        = "direto"
ident_sd["DESC_FIO"]      = 0.0
ident_sd["AJUSTE_1A_PASSADA_BRL"] = 0.0
ident_sd["AJUSTE_2A_PASSADA_BRL"] = 0.0
ident_sd["FAIXA1_PCT"] = np.nan
ident_sd["RECEITA_APE_BRL"] = 0.0
ident_sd["AJUSTE_APE_BRL"] = 0.0
ident_sd["TARIFAVEL_SD"]  = "SIM"

# the SD needs no adjustment: block 2 equals block 1
B2_SD_CAMADAS = {c: B1_SD_CAMADAS[c].copy() for c in CAMADAS}

print(f"linhas do SD prontas: {len(ident_sd)}")
print(f"  receita  R$ {ident_sd['RECEITA_BRL'].sum():,.2f}")
print(f"  calculado R$ {ident_sd['CALCULADO_TA'].sum():,.2f}")
print(f"  diferenca R$ {ident_sd['DIFERENCA_BRL'].sum():,.2f}  "
      f"(linhas tarifaveis; tributos nao tem abertura)")
print()
print(ident_sd[["CLASSE_SUBCLASSE", "FLAG_MERCADO", "GRP_MONT", "DEMANDA",
                "TOTAL_CONSUMO", "RECEITA_BRL", "CALCULADO_TA"]].head(12).to_string())

---
## Passo 14b — Conferências do SD

1. A abertura reproduz a receita, linha a linha.
2. O total do SD, descontados PIS, COFINS e ultrapassagem, bate com a aba consolidada no mesmo mês de
   referência, cliente a cliente.

In [ ]:
ok_sd = True
tar = ident_sd["TARIFAVEL_SD"] == "SIM"

# 1. a abertura reproduz a receita, linha a linha
d = (ident_sd.loc[tar, "CALCULADO_TA"] - ident_sd.loc[tar, "RECEITA_BRL"]).abs().max()
print(f"1. SD: abertura x receita, linha a linha ... diferenca maxima {d:,.6f}")
ok_sd &= d < 0.01

# 2. o total bate com a aba consolidada, que e a que vai para a contabilidade
# a aba consolidada soma PIS e COFINS na receita; aqui o importe e limpo
no_mes = sd_consolidado[sd_consolidado["REFERENCIA"] == alvo]
def _cons(info):
    return no_mes.loc[no_mes["INFORMACAO"] == info, "VALOR"].sum()
bruto, pis, cofins = _cons("Receita"), _cons("PIS"), _cons("COFINS")
fora_regra = sd_fora["RECEITA_BRL"].sum() if len(sd_fora) else 0.0
liquido = bruto - pis - cofins - fora_regra
soma = ident_sd["RECEITA_BRL"].sum()
print(f"2. SD: total x aba consolidada")
print(f"   consolidado bruto ........ R$ {bruto:,.2f}")
print(f"   menos PIS ................ R$ {pis:,.2f}")
print(f"   menos COFINS ............. R$ {cofins:,.2f}")
print(f"   menos ultrapassagem ...... R$ {fora_regra:,.2f}   (fora da abertura; entra como nao tarifavel)")
print(f"   base comparavel .......... R$ {liquido:,.2f}")
print(f"   pela abertura ............ R$ {soma:,.2f}")
print(f"   diferenca ................ R$ {soma - liquido:,.2f}")
ok_sd &= abs(soma - liquido) < 1.0
# client by client: where a difference would come from
_rec = ident_sd.groupby("CLASSE_SUBCLASSE")["RECEITA_BRL"].sum()
_ult = sd_fora.groupby("CLIENTE")["RECEITA_BRL"].sum() if len(sd_fora) else pd.Series(dtype=float)
_cli = no_mes.pivot_table(index="CLIENTE", columns="INFORMACAO", values="VALOR", aggfunc="sum").fillna(0.0)
for _c in ("Receita", "PIS", "COFINS"):
    if _c not in _cli: _cli[_c] = 0.0
_dif = (_rec.reindex(_cli.index).fillna(0.0) + _ult.reindex(_cli.index).fillna(0.0)
        - (_cli["Receita"] - _cli["PIS"] - _cli["COFINS"]))
for _c, _v in _dif[_dif.abs() > 1.0].items():
    print(f"   cliente com diferenca: {_c} R$ {_v:,.2f}")

# 3. SD lines without a tariff (PIS, COFINS and the CERIS demand overrun) join the BW non-tariff
#    lines: they reach the Componentes total and the Receita Faturada layout (PIS/COFINS and UD/ER)
_base = dict(LINHA_EXCEL=np.nan, CATEGORIA_TARIFA="", GRP_QUANT="", GRP_QUANT_TXT="",
             OPERACAO_TXT="Faturamento SD", SUBOPERACAO="", SUBOPERACAO_TXT="mês de competência",
             QTD_CONSUMO_TUSD=0.0, QTD_CONSUMO_TE=0.0, QTD_DEMANDA_TUSD=0.0, QTD_DEMANDA=0.0,
             DEMANDA_KW=0.0, TOTAL_CONSUMO_XLS=0.0, TOTAL_DEMANDA_XLS=0.0,
             TARIFAVEL=False, SITUACAO="NÃO", TIPO_FATURAMENTO="FAT")
_nt = []
for _cliente, _r in _cli.iterrows():
    _tipo = DE_PARA_CLIENTE.get(_cliente)
    for _info in ("PIS", "COFINS"):
        if _tipo and _r[_info]:
            _nt.append(dict(_base, FLAG_MERCADO=_tipo, CLASSE_PRINCIPAL=_tipo, CLASSE_SUBCLASSE=_cliente,
                            GRP_MONT=f"{_info} - SD", OPERACAO="R/#", RECEITA_BRL=_r[_info]))
for _, _r in sd_fora.iterrows():
    _nt.append(dict(_base, FLAG_MERCADO=_r["TIPO"], CLASSE_PRINCIPAL=_r["TIPO"],
                    CLASSE_SUBCLASSE=_r["CLIENTE"], GRP_MONT=f"{_r['RUBRICA']} (demanda, kW) - SD",
                    OPERACAO=_r["COMPONENTE"], DEMANDA_KW=_r["QUANTIDADE"], RECEITA_BRL=_r["RECEITA_BRL"]))
if _nt:
    sd_nao_tarifavel = pd.DataFrame(_nt).reindex(columns=fora.columns)
    fora = pd.concat([fora, sd_nao_tarifavel], ignore_index=True)
    print(f"3. SD sem tarifa (PIS, COFINS, ultrapassagem) somado aos nao tarifaveis: "
          f"R$ {sd_nao_tarifavel['RECEITA_BRL'].sum():,.2f}")
_total_sd = soma + (sd_nao_tarifavel["RECEITA_BRL"].sum() if _nt else 0.0)
print(f"   SD total no pxq R$ {_total_sd:,.2f} | consolidado bruto R$ {bruto:,.2f} | "
      f"diferenca R$ {_total_sd - bruto:,.2f}")

# clientes presentes num lado e ausentes no outro, que e onde mora a diferenca
lado_a = set(ident_sd["CLASSE_SUBCLASSE"])
lado_b = set(no_mes.loc[no_mes["INFORMACAO"] == "Receita", "CLIENTE"])
if lado_a - lado_b: print("   so nas abas de cliente:", ", ".join(sorted(lado_a - lado_b)))
if lado_b - lado_a: print("   so no consolidado     :", ", ".join(sorted(lado_b - lado_a)))

print("\nCONFERENCIAS DO SD PASSARAM" if ok_sd else "\nALGUMA CONFERENCIA DO SD FALHOU")

---
## Passo 15 — Gerar a planilha

A saída `1-Entrega/pxq/pxq_<ref>.xlsx` traz:

| Aba | Conteúdo |
|---|---|
| **Componentes** | Os 37 itens na tarifa de aplicação, blocos 1 e 2, mais os totais de BE, CVA e FIN do bloco 2 |
| **Componentes_BE** | Os mesmos itens na base econômica (alimenta o layout Receita Faturada) |
| **Familias** | As sete famílias (Fio A, Fio B, Encargos, Perdas, Energia, RI, Outros), nas 4 camadas, blocos 1 e 2 |
| **Nao_tarifaveis_linha** | Cada lançamento não tarifável do BW, linha a linha, com classe e mercado |
| **Calibracao_subgrupo**, **Calibracao_classe** | Só nos meses de reajuste: peso estimado de cada vigência e conferência por classe (Passo 7b) |
| **Diferencas_linha**, **Diferencas_grupo** | Só com `APLICAR_2A_PASSADA = False`; o arquivo leva o sufixo `_sem_2a_passada` |

Nas abas de itens e famílias, os lançamentos fora da conta entram no fim, com os blocos vazios, para
que a soma da receita bata com o total do BW. Eles vêm marcados em `Tarifável?`:

- **NÃO** — sem tarifa homologada por item (tributos, bandeiras, reativos, ultrapassagem,
  substituição tributária etc.). São agrupados por categoria, mercado, tipo de lançamento e tipo
  de faturamento.
- **SEM TARIFA** — lançamento de fornecimento cuja chave não existe na tabela de tarifas. Merece
  investigação: ou o cadastro está errado, ou falta uma linha na tabela.

O passo seguinte, fora do notebook, é `python 3-Py/1-Entrega/gera_receita_faturada.py <ref>`, que converte
esta planilha no layout Receita Faturada.

In [ ]:
IDENTIFICACAO = ["CHAVE", "CLASSE_PRINCIPAL", "CATEGORIA_TARIFA", "GRP_MONT", "GRP_QUANT",
                 "GRP_QUANT_TXT", "OPERACAO", "OPERACAO_TXT", "SUBOPERACAO", "SUBOPERACAO_TXT",
                 "FLAG_MERCADO", "CLASSE_SUBCLASSE", "TIPO_FATURAMENTO", "LINHAS_ORIGEM",
                 "CONSUMO_TUSD", "CONSUMO_TE", "DEMANDA", "TOTAL_CONSUMO", "TOTAL_DEMANDA",
                 "RECEITA_BRL", "CALCULADO_TA", "DIFERENCA_BRL", "PCT_AJUSTE"]
ROTULO = {"SOURCE": "Source", "CHAVE": "Chave", "CLASSE_PRINCIPAL": "Classe principal",
          "CATEGORIA_TARIFA": "Categoria tarifa", "GRP_MONT": "Grp.estatísts.mont.",
          "GRP_QUANT": "Grp.estatísts.quant.", "GRP_QUANT_TXT": "Grp.estatísts.quant. TEXTO",
          "OPERACAO": "Operação principal", "OPERACAO_TXT": "Operação principal TEXTO",
          "SUBOPERACAO": "Suboperação", "SUBOPERACAO_TXT": "Suboperação TEXTO",
          "FLAG_MERCADO": "Flag Cli.Liv./ Cat.", "CLASSE_SUBCLASSE": "Classe/Sub-classe",
          "TIPO_FATURAMENTO": "Tipo de faturamento",
          "LINHAS_ORIGEM": "Linhas de origem", "CONSUMO_TUSD": "Consumo TUSD (kWh)",
          "CONSUMO_TE": "Consumo TE (kWh)", "DEMANDA": "Demanda (kW)",
          "TOTAL_CONSUMO": "TOTAL CONSUMO", "TOTAL_DEMANDA": "TOTAL DEMANDA",
          "RECEITA_BRL": "RECEITA DE FORNECIMENTO",
          "CALCULADO_TA": "VALOR CALCULADO (bloco 1)", "DIFERENCA_BRL": "DIFERENÇA (R$)",
          "PCT_AJUSTE": "DIFERENÇA (%)", "REGIME": "Regime de ajuste",
          "DESC_FIO": "% de desconto no fio",
          "AJUSTE_1A_PASSADA_BRL": "Ajuste 1ª passada (R$)",
          "AJUSTE_2A_PASSADA_BRL": "Ajuste 2ª passada (R$)",
          "FAIXA1_PCT": "% energia na faixa 01 (estimada pela receita)",
          "RECEITA_APE_BRL": "Receita APE incluída (desconto + complemento, R$)",
          "AJUSTE_APE_BRL": "Ajuste regime APE (R$)",
          "TARIFAVEL_TXT": "Tarifável?"}

def familias_por_camada(bloco):
    """Soma os componentes de cada familia, em cada camada."""
    saida = {}
    for camada in CAMADAS:
        for fam, lista in FAMILIA.items():
            saida[f"{camada} · {fam}"] = bloco[camada][lista].sum(axis=1)
    return pd.DataFrame(saida)

def monta_aba(colunas_b1, colunas_b2, extra=None, sd_b1=None, sd_b2=None, sd_extra=None,
              extra_b1=None, sd_extra_b1=None):
    esq = ident[IDENTIFICACAO].copy()
    esq.insert(0, "SOURCE", "BW")
    esq["TARIFAVEL_TXT"] = "SIM"
    corpo = pd.concat([esq.reset_index(drop=True),
                       colunas_b1.add_prefix("[1] ").reset_index(drop=True),
                       ident[["REGIME", "RECEITA_APE_BRL", "AJUSTE_APE_BRL", "DESC_FIO", "AJUSTE_1A_PASSADA_BRL", "AJUSTE_2A_PASSADA_BRL", "FAIXA1_PCT"]].reset_index(drop=True),
                       colunas_b2.add_prefix("[2] ").reset_index(drop=True)], axis=1)
    if extra is not None:
        corpo = pd.concat([corpo, extra.add_prefix("[2] ").reset_index(drop=True)], axis=1)
    if extra_b1 is not None:
        # bloco-1 layer totals: let the bloco 1 x bloco 2 check split CVA and FIN
        corpo = pd.concat([corpo, extra_b1.add_prefix("[1] ").reset_index(drop=True)], axis=1)

    # linhas fora da conta, no fim da lista. Agrupadas mais grosso, porque nelas
    # o que importa e o tipo de lancamento, nao o detalhe de classe e operacao.
    g2 = fora.groupby(["SITUACAO", "CATEGORIA_TARIFA", "FLAG_MERCADO", "GRP_MONT", "TIPO_FATURAMENTO"],
                      dropna=False, sort=False)
    nt = g2.agg(**{c: (c, junta) for c in ["CLASSE_PRINCIPAL", "CLASSE_SUBCLASSE", "GRP_QUANT",
                                           "GRP_QUANT_TXT", "OPERACAO", "OPERACAO_TXT",
                                           "SUBOPERACAO", "SUBOPERACAO_TXT"]})
    nt["LINHAS_ORIGEM"] = g2["LINHA_EXCEL"].agg(lista_linhas)
    nt["CONSUMO_TUSD"] = g2["QTD_CONSUMO_TUSD"].sum()
    nt["CONSUMO_TE"]   = g2["QTD_CONSUMO_TE"].sum()
    nt["DEMANDA"]      = g2["DEMANDA_KW"].sum()
    nt["TOTAL_CONSUMO"] = g2["TOTAL_CONSUMO_XLS"].sum()
    nt["TOTAL_DEMANDA"] = g2["TOTAL_DEMANDA_XLS"].sum()
    nt["RECEITA_BRL"]  = g2["RECEITA_BRL"].sum()
    nt = nt.reset_index()
    nt["CHAVE"] = np.where(nt["SITUACAO"] == "SEM TARIFA", "(chave sem tarifa na tabela de tarifas)", "")
    nt["TARIFAVEL_TXT"] = nt["SITUACAO"]
    nt["SOURCE"] = "BW"
    nt = nt.reindex(columns=corpo.columns)
    nt[[c for c in corpo.columns if c.startswith("[")]] = np.nan
    nt[["CALCULADO_TA", "DIFERENCA_BRL", "PCT_AJUSTE", "REGIME", "DESC_FIO",
     "AJUSTE_1A_PASSADA_BRL", "AJUSTE_2A_PASSADA_BRL", "FAIXA1_PCT", "RECEITA_APE_BRL", "AJUSTE_APE_BRL"]] = np.nan

    partes = [corpo, nt]

    # linhas do SD, quando o bloco da Parte II foi executado
    if sd_b1 is not None:
        esq_sd = ident_sd[IDENTIFICACAO].copy()
        esq_sd.insert(0, "SOURCE", "SD")
        esq_sd["TARIFAVEL_TXT"] = ident_sd["TARIFAVEL_SD"].values
        bloco_sd = pd.concat([esq_sd.reset_index(drop=True),
                              sd_b1.add_prefix("[1] ").reset_index(drop=True),
                              ident_sd[["REGIME", "RECEITA_APE_BRL", "AJUSTE_APE_BRL", "DESC_FIO", "AJUSTE_1A_PASSADA_BRL", "AJUSTE_2A_PASSADA_BRL", "FAIXA1_PCT"]].reset_index(drop=True),
                              sd_b2.add_prefix("[2] ").reset_index(drop=True)], axis=1)
        if sd_extra is not None:
            bloco_sd = pd.concat([bloco_sd, sd_extra.add_prefix("[2] ").reset_index(drop=True)], axis=1)
        if sd_extra_b1 is not None:
            bloco_sd = pd.concat([bloco_sd, sd_extra_b1.add_prefix("[1] ").reset_index(drop=True)], axis=1)
        bloco_sd = bloco_sd.reindex(columns=corpo.columns)
        partes.append(bloco_sd)

    return pd.concat(partes, ignore_index=True).rename(columns=ROTULO)

aba_componentes = monta_aba(
    B1_GRUPO["TA"][COMPONENTES],
    B2_GRUPO["TA"][COMPONENTES],
    extra=pd.DataFrame({f"Total {c}": B2_GRUPO[c][COMPONENTES].sum(axis=1)
                        for c in ["TA", "BE", "CVA", "FIN"]}),
    sd_b1=B1_SD_CAMADAS["TA"][COMPONENTES], sd_b2=B2_SD_CAMADAS["TA"][COMPONENTES],
    sd_extra=pd.DataFrame({f"Total {c}": B2_SD_CAMADAS[c][COMPONENTES].sum(axis=1)
                           for c in ["TA", "BE", "CVA", "FIN"]}, index=B1_SD.index))
# same layout in the BE layer: feeds the Receita Faturada model (component columns are BE)
_totais = pd.DataFrame({f"Total {c}": B2_GRUPO[c][COMPONENTES].sum(axis=1) for c in ["TA", "BE", "CVA", "FIN"]})
_totais_b1 = pd.DataFrame({f"Total {c}": B1_GRUPO[c][COMPONENTES].sum(axis=1) for c in ["TA", "BE", "CVA", "FIN"]})
aba_componentes_be = monta_aba(
    B1_GRUPO["BE"][COMPONENTES], B2_GRUPO["BE"][COMPONENTES], extra=_totais,
    sd_b1=B1_SD_CAMADAS["BE"][COMPONENTES], sd_b2=B2_SD_CAMADAS["BE"][COMPONENTES],
    sd_extra=pd.DataFrame({f"Total {c}": B2_SD_CAMADAS[c][COMPONENTES].sum(axis=1)
                           for c in ["TA", "BE", "CVA", "FIN"]}, index=B1_SD.index),
    extra_b1=_totais_b1,
    sd_extra_b1=pd.DataFrame({f"Total {c}": B1_SD_CAMADAS[c][COMPONENTES].sum(axis=1)
                              for c in ["TA", "BE", "CVA", "FIN"]}, index=B1_SD.index))
aba_familias = monta_aba(familias_por_camada(B1_GRUPO), familias_por_camada(B2_GRUPO),
                         sd_b1=familias_por_camada(B1_SD_CAMADAS),
                         sd_b2=familias_por_camada(B2_SD_CAMADAS))

print(f"aba Componentes: {aba_componentes.shape[0]} linhas x {aba_componentes.shape[1]} colunas")
print(f"aba Familias   : {aba_familias.shape[0]} linhas x {aba_familias.shape[1]} colunas")

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

def formata(planilha, n_ident):
    planilha.freeze_panes = planilha.cell(row=2, column=n_ident + 1)
    for celula in planilha[1]:
        celula.font = Font(bold=True, size=9, color="FFFFFF")
        celula.fill = PatternFill("solid", fgColor="1F4E79")
        celula.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
    planilha.row_dimensions[1].height = 42
    for col in range(1, planilha.max_column + 1):
        letra = get_column_letter(col)
        planilha.column_dimensions[letra].width = 34 if col == 1 else (
            22 if col <= n_ident else 16)
        if col > n_ident:
            for linha in range(2, planilha.max_row + 1):
                planilha.cell(row=linha, column=col).number_format = "#,##0.00"
    planilha.auto_filter.ref = (f"A1:{get_column_letter(planilha.max_column)}"
                                f"{planilha.max_row}")

# "Leia-me" sheet: what each sheet holds and where it is used
LEIA_ME = [
    ("Componentes", "Uma linha por grupo do BW (chave tarifária x classe x operação x tipo de faturamento: "
     "FAT, REFAT ou CNR) e por linha do SD (sempre FAT). "
     "Colunas \"[1] <item>\": bloco 1 (quantidade x tarifa homologada) na tarifa de aplicação (TA), item a item. "
     "Colunas \"[2] <item>\": bloco 2 (entrega, fecha na receita). \"[2] Total TA/BE/CVA/FIN\": totais do bloco 2 "
     "por camada. Colunas de ajuste: regime APE, 1ª passada (fonte incentivada) e 2ª passada.",
     "3-Py/1-Entrega/gera_receita_faturada.py (pesos dos tributos e resíduo); "
     "3-Py/2-Checks/gera_receita_faturada_bl1_vs_bl2.py; 3-Py/2-Checks/check_mr_bloco2.py; "
     "checks do SAMP (lib_samp.py); notebook do RNF (Abertura_RNF.ipynb)"),
    ("Componentes_BE", "Mesmas linhas e colunas de Componentes, na base econômica (BE). No fim, os totais por "
     "camada dos dois blocos: \"[2] Total TA/BE/CVA/FIN\" (bloco 2) e \"[1] Total TA/BE/CVA/FIN\" (bloco 1).",
     "3-Py/1-Entrega/gera_receita_faturada.py (colunas I a Z do layout Receita Faturada, pelo bloco 2); "
     "3-Py/2-Checks/gera_receita_faturada_bl1_vs_bl2.py (layout pelo bloco 1 e totais por camada do bloco 1)"),
    ("Familias", "Os itens agrupados nas sete famílias (Fio A, Fio B, encargos, perdas, energia, RI, outros), "
     "por camada (TA, BE, CVA, FIN), nos blocos 1 e 2.",
     "3-Py/2-Checks/check_mr_camadas.py; 3-Py/2-Checks/check_mr_bloco2.py; checks do SAMP (lib_samp.py)"),
    ("Nao_tarifaveis_linha", "Os lançamentos do BW sem tarifa homologada (tributos, bandeiras, reativos, "
     "ultrapassagem, acertos), um a um, com a classe e o tipo de faturamento de cada lançamento.",
     "3-Py/1-Entrega/gera_receita_faturada.py (colunas Bandeira, UD/ER, Outros, PIS/COFINS e ICMS)"),
    ("Calibracao_subgrupo, Calibracao_classe", "Só nos meses de reajuste: o peso estimado de cada vigência por "
     "subgrupo e a conferência por classe.", "Conferência da tarifa proporcional (procedimento, Passo 7b)"),
    ("Diferencas_linha, Diferencas_grupo", "Só com APLICAR_2A_PASSADA = False (arquivo _sem_2a_passada): a "
     "diferença que sobra antes da 2ª passada, linha a linha do BW e por grupo.", "Investigação de diferenças"),
]
leia_me = pd.DataFrame(LEIA_ME, columns=["Aba", "O que contém", "Onde é usada"])

with pd.ExcelWriter(SAIDA, engine="openpyxl") as escritor:
    leia_me.to_excel(escritor, sheet_name="Leia-me", index=False)
    _lm = escritor.book["Leia-me"]
    for _col, _larg in zip("ABC", (34, 110, 80)):
        _lm.column_dimensions[_col].width = _larg
    for _lin in _lm.iter_rows(min_row=1):
        for _cel in _lin:
            _cel.alignment = Alignment(wrap_text=True, vertical="top")
    for _cel in _lm[1]:
        _cel.font = Font(bold=True)
    aba_componentes.to_excel(escritor, sheet_name="Componentes", index=False)
    aba_familias.to_excel(escritor, sheet_name="Familias", index=False)
    aba_componentes_be.to_excel(escritor, sheet_name="Componentes_BE", index=False)
    n_ident = len(IDENTIFICACAO) + 2
    formata(escritor.book["Componentes"], n_ident)
    formata(escritor.book["Familias"], n_ident)
    formata(escritor.book["Componentes_BE"], n_ident)
    # a coluna de percentual em formato de porcentagem
    for nome in ("Componentes", "Familias", "Componentes_BE"):
        planilha = escritor.book[nome]
        for col in range(1, planilha.max_column + 1):
            if planilha.cell(row=1, column=col).value in (ROTULO["PCT_AJUSTE"],
                                                          ROTULO["DESC_FIO"]):
                for linha in range(2, planilha.max_row + 1):
                    planilha.cell(row=linha, column=col).number_format = "0.00%"

if not APLICAR_2A_PASSADA:
    # gap sheets: line by line (with the group number to sum the TUSD/TE pair) and by group
    ROT_DIF = {"GRUPO": "Grupo", "LINHA_EXCEL": "Linha na base agregada do BW", "CATEGORIA_TARIFA": "Categoria tarifa",
               "FLAG_MERCADO": "Flag Cli.Liv./ Cat.", "CLASSE_PRINCIPAL": "Classe principal",
               "CLASSE_SUBCLASSE": "Classe/Sub-classe", "GRP_MONT": "Grp.estatísts.mont.",
               "GRP_QUANT": "Grp.estatísts.quant.", "GRP_QUANT_TXT": "Grp.estatísts.quant. TEXTO",
               "OPERACAO": "Operação principal", "OPERACAO_TXT": "Operação principal TEXTO",
               "SUBOPERACAO": "Suboperação", "SUBOPERACAO_TXT": "Suboperação TEXTO", "CHAVE": "Chave",
               "TIPO_FATURAMENTO": "Tipo de faturamento",
               "QTD_CONSUMO_TUSD": "Consumo TUSD (kWh)", "QTD_CONSUMO_TE": "Consumo TE (kWh)",
               "DEMANDA_KW": "Demanda (kW)", "RECEITA_BRL": "RECEITA DE FORNECIMENTO",
               "REGIME": "Regime de ajuste", "BLOCO1_TA": "VALOR CALCULADO (bloco 1)",
               "DESC_FIO": "% de desconto no fio", "AJUSTE_1A_PASSADA_BRL": "Ajuste 1ª passada (R$)",
               "BLOCO2_TA": "Bloco 2 sem 2ª passada (R$)", "DIFERENCA_BRL": "DIFERENÇA NÃO AJUSTADA (R$)"}
    linhas = DIF_LINHAS.sort_values(["GRUPO", "LINHA_EXCEL"]).rename(columns=ROT_DIF)
    grupos = ident[["CHAVE", "CATEGORIA_TARIFA", "FLAG_MERCADO", "CLASSE_SUBCLASSE", "TIPO_FATURAMENTO",
                    "GRP_QUANT", "LINHAS_ORIGEM",
                    "REGIME", "RECEITA_BRL", "CALCULADO_TA", "DESC_FIO", "AJUSTE_1A_PASSADA_BRL",
                    "DIF_NAO_AJUSTADA_BRL"]].copy()
    grupos.insert(0, "GRUPO", np.arange(1, len(grupos) + 1))
    grupos = grupos.rename(columns={**ROT_DIF, "LINHAS_ORIGEM": "Linhas de origem",
                                    "CALCULADO_TA": "VALOR CALCULADO (bloco 1)",
                                    "DIF_NAO_AJUSTADA_BRL": "DIFERENÇA NÃO AJUSTADA (R$)"})
    with pd.ExcelWriter(SAIDA, engine="openpyxl", mode="a") as escritor:
        linhas.to_excel(escritor, sheet_name="Diferencas_linha", index=False)
        grupos.to_excel(escritor, sheet_name="Diferencas_grupo", index=False)
        for nome, n_id in (("Diferencas_linha", 15), ("Diferencas_grupo", 8)):
            formata(escritor.book[nome], n_id)
            planilha = escritor.book[nome]
            for col in range(1, planilha.max_column + 1):
                if planilha.cell(row=1, column=col).value == ROT_DIF["DESC_FIO"]:
                    for linha in range(2, planilha.max_row + 1):
                        planilha.cell(row=linha, column=col).number_format = "0.00%"
    print(f"abas de diferenca: {len(linhas)} linhas do BW | {len(grupos)} grupos | "
          f"total R$ {DIF_LINHAS['DIFERENCA_BRL'].sum():,.2f}")

# non-tariff lines, one row per BW line: lets 3-Py/1-Entrega/gera_receita_faturada.py open taxes and
# tariff flags by class (the grouped Componentes sheet concatenates classes in these groups)
COLS_NT = {"LINHA_EXCEL": "Linha na base agregada do BW", "FLAG_MERCADO": "Flag Cli.Liv./ Cat.",
           "CLASSE_PRINCIPAL": "Classe principal", "CLASSE_SUBCLASSE": "Classe/Sub-classe",
           "CATEGORIA_TARIFA": "Categoria tarifa", "GRP_MONT": "Grp.estatísts.mont.",
           "GRP_QUANT": "Grp.estatísts.quant.", "OPERACAO": "Operação principal",
           "SUBOPERACAO": "Suboperação", "TIPO_FATURAMENTO": "Tipo de faturamento",
           "RECEITA_BRL": "RECEITA DE FORNECIMENTO",
           "SITUACAO": "Tarifável?"}
nao_tarifaveis = fora[list(COLS_NT)].rename(columns=COLS_NT)
with pd.ExcelWriter(SAIDA, engine="openpyxl", mode="a") as escritor:
    nao_tarifaveis.to_excel(escritor, sheet_name="Nao_tarifaveis_linha", index=False)
    formata(escritor.book["Nao_tarifaveis_linha"], len(COLS_NT) - 1)

if RELATORIO_CALIBRACAO is not None:
    with pd.ExcelWriter(SAIDA, engine="openpyxl", mode="a") as escritor:
        RELATORIO_CALIBRACAO.to_excel(escritor, sheet_name="Calibracao_subgrupo", index=False)
        CONFERENCIA_CALIBRACAO.to_excel(escritor, sheet_name="Calibracao_classe", index=False)
        for nome in ("Calibracao_subgrupo", "Calibracao_classe"):
            formata(escritor.book[nome], 1)

print(f"gravado: {SAIDA}")
print(f"referencia {REFERENCIA} | " +
      ", ".join(f"{ano} ({100*p:.1f}%)" for ano, p in VIGENCIAS_NO_MES) +
      (f" | peso estimado: {', '.join(PESO_ESTIMADO)}" if PESO_ESTIMADO else ""))